# APPLICATION OF XGBOOST FROM APRIL 2026

In [ ]:
# # ============================================================
# # 🔍 PREDICT SECTION & CLUSTER — Device: LL1801
# # ============================================================

# import pandas as pd
# import numpy as np
# import re, pickle, os, warnings
# warnings.filterwarnings('ignore')

# # ── 1. LOAD MODELS & ENCODERS ────────────────────────────────
# SAVE_DIR = r'C:\Users\sitisyaziyah\Documents\Device_Identifier\Section\XGBoost16042026'

# with open(os.path.join(SAVE_DIR, 'model_section.pkl'), 'rb') as f:
#     model_section = pickle.load(f)

# with open(os.path.join(SAVE_DIR, 'model_cluster.pkl'), 'rb') as f:
#     model_cluster = pickle.load(f)

# with open(os.path.join(SAVE_DIR, 'label_encoders.pkl'), 'rb') as f:
#     enc           = pickle.load(f)

# le_customer    = enc['customer']
# le_project     = enc['project']
# le_prefix      = enc['prefix']
# le_suffix_lt   = enc['suffix_lt']
# le_suffix_last = enc['suffix_last']
# le_section     = enc['section']
# le_cluster     = enc['cluster']
# feature_cols   = enc['feature_columns']

# print(f"✅ Models loaded | Sections: {list(le_section.classes_)} | Clusters: {list(le_cluster.classes_)}")

# # ── 2. FEATURE ENGINEERING HELPERS ──────────────────────────
# def extract_numeric_block(did):
#     m = re.search(r'\d+', str(did))
#     return int(m.group()) if m else -1

# def extract_prefix(did):
#     m = re.match(r'^([A-Za-z]+)', str(did))
#     return m.group(1).upper() if m else ''

# def extract_suffix_lt(did):
#     m = re.search(r'\d+([A-Za-z]*)$', str(did))
#     return m.group(1).upper() if m else ''

# def extract_suffix_full(did):
#     m = re.search(r'\d+(.*)$', str(did))
#     return m.group(1) if m else ''

# def safe_encode(encoder, value, default=0):
#     try:
#         return int(encoder.transform([value])[0])
#     except ValueError:
#         print(f"  ⚠ Unseen label '{value}' → using default={default}")
#         return default

# import pandas as pd

# df_oiltek_num = pd.read_pickle(
#     r"C:\Users\sitisyaziyah\Documents\Device_Identifier\Section\XGBoost16042026\df_oiltek_num.pkl"
# )

# # ── 3. INPUT ─────────────────────────────────────────────────
# device_id = 'LL510'
# customer  = 'OILTEK'
# project   = 'A1706'

# # ── 4. COMPUTE numeric_block_rank (PURE NUMERIC COMPARISON) ──
# # ── 4. COMPUTE numeric_block_rank (FROM df_oiltek_num) ───────

# numeric_block = extract_numeric_block(device_id)

# # Filter relevant subset
# df_ref = df_oiltek_num[
#     (df_oiltek_num["CUSTOMER"] == customer) &
#     (df_oiltek_num["PROJECT"] == "A1706")
# ]

# if not df_ref.empty:
#     # Ensure numeric_block column is clean
#     df_ref = df_ref.copy()
#     df_ref["numeric_block"] = pd.to_numeric(df_ref["numeric_block"], errors="coerce")

#     # Get sorted unique numeric blocks
#     unique_blocks = sorted(df_ref["numeric_block"].dropna().unique())

#     # PURE ranking
#     numeric_block_rank = sum(b <= numeric_block for b in unique_blocks)

#     # Edge case: if all values are bigger
#     if numeric_block_rank == 0:
#         numeric_block_rank = 1

#     print(f"  ✅ Numeric-based rank (df reference): {numeric_block} → rank {numeric_block_rank}")

# else:
#     numeric_block_rank = 1
#     print("  ⚠ No reference data found — using rank=1 (fallback)")

In [ ]:
# # ── 5. BUILD FEATURE VECTOR ──────────────────────────────────
# pfx        = extract_prefix(device_id)
# sfx_lt     = extract_suffix_lt(device_id)
# sfx_full   = extract_suffix_full(device_id)
# sfx_last   = sfx_full[-1] if sfx_full else ''


# # ── 5. BUILD FEATURE VECTOR (NUMERIC-ONLY VERSION) ───────────
# fv = {
#     'customer_enc'       : safe_encode(le_customer, customer),
#     'project_enc'        : safe_encode(le_project, project),
#     'numeric_block'      : numeric_block,
#     'numeric_block_rank' : numeric_block_rank,
#     'device_id_length'   : len(device_id),
#     'has_numeric'        : int(numeric_block != -1),
#     'equip_id_length'    : len(device_id),
#     'equip_id_digit_count': len(re.findall(r'\d', device_id)),
#     'equip_id_letter_count': len(re.findall(r'[A-Za-z]', device_id)),
# }

# X_new = pd.DataFrame([fv])

# X_new = pd.DataFrame([fv])

# # Ensure all expected columns exist
# for col in feature_cols:
#     if col not in X_new.columns:
#         X_new[col] = 0

# # Reorder columns
# X_new = X_new[feature_cols]


In [ ]:
# # ── 6. PREDICT ──────────────────────────────────────────────
# assert list(X_new.columns) == list(feature_cols), "❌ Feature mismatch with training!"

# try:
#     sec_proba = model_section.predict_proba(X_new)[0]
#     clu_proba = model_cluster.predict_proba(X_new)[0]
# except Exception as e:
#     print(f"❌ Prediction failed: {e}")
#     raise

# sec_pred = le_section.classes_[np.argmax(sec_proba)]
# clu_pred = le_cluster.classes_[np.argmax(clu_proba)]

# sec_conf = np.max(sec_proba)
# clu_conf = np.max(clu_proba)


# # ── 7. DISPLAY RESULTS ──────────────────────────────────────
# print("\n" + "=" * 65)
# print(f"  PREDICTION RESULT — {device_id} | {customer} | {project}")
# print("=" * 65)
# print(f"  ✅ Section  →  {sec_pred}   ({sec_conf:.2%})")
# print(f"  ✅ Cluster  →  {clu_pred}   ({clu_conf:.2%})")

# print("\n  SECTION probabilities:")
# sec_df = pd.DataFrame({'Section': le_section.classes_, 'Prob': sec_proba})
# sec_df = sec_df.sort_values('Prob', ascending=False).reset_index(drop=True)

# for _, row in sec_df.iterrows():
#     bar = '█' * int(row['Prob'] * 40)
#     tag = '<< PREDICTED' if row['Section'] == sec_pred else ''
#     print(f"    {row['Section']:<12} {row['Prob']:6.2%}  {bar} {tag}")

# print("\n  CLUSTER probabilities:")
# clu_df = pd.DataFrame({'Cluster': le_cluster.classes_, 'Prob': clu_proba})
# clu_df = clu_df.sort_values('Prob', ascending=False).reset_index(drop=True)

# for _, row in clu_df.iterrows():
#     bar = '█' * int(row['Prob'] * 40)
#     tag = '<< PREDICTED' if row['Cluster'] == clu_pred else ''
#     print(f"    {row['Cluster']:<12} {row['Prob']:6.2%}  {bar} {tag}")

# print("\n" + "=" * 65)

In [ ]:
# print(df_oiltek_num[(df_oiltek_num["PROJECT"] == "A1706")& (df_oiltek_num["SECTION"] == "SECTION 1") & (df_oiltek_num["CLUSTER"] == "CLUSTER 1")])

# NEW SCRIPT

In [ ]:
# # ============================================================
# # 🤖 DEVICE CLUSTER — MODEL APPLICATION SCRIPT
# # ============================================================
# # Loads saved pipeline (models + encoders) from training phase
# # and predicts SECTION + CLUSTER for new device entries.
# #
# # Threshold: confidence < 50% → labelled as UNKNOWN
# # ============================================================

# import pandas as pd
# import numpy as np
# import re
# import pickle
# import os
# import warnings
# warnings.filterwarnings('ignore')


# # ============================================================
# # 📂 PATHS — Update to match your saved model directory
# # ============================================================
# MODEL_DIR       = r"C:\Users\sitisyaziyah\source\repos\DeviceCluster\1.training_model\model_config"
# MODEL_DIR_OUTPUT_PREDICTION = r"C:\Users\sitisyaziyah\source\repos\DeviceCluster\1.training_model\output_prediction"


# UNKNOWN_THRESHOLD = 0.50   # confidence below this → UNKNOWN


# # ============================================================
# # 🔧 FEATURE ENGINEERING FUNCTIONS (must mirror training)
# # ============================================================

# def extract_numeric_block(device_id):
#     match = re.search(r'\d+', str(device_id))
#     return int(match.group()) if match else -1

# def extract_prefix(device_id):
#     match = re.match(r'^([A-Za-z]+)', str(device_id))
#     return match.group(1).upper() if match else ''

# def extract_suffix_letters(device_id):
#     match = re.search(r'\d+([A-Za-z]*)$', str(device_id))
#     return match.group(1).upper() if match else ''

# def extract_suffix_full(device_id):
#     match = re.search(r'\d+(.*)$', str(device_id))
#     return match.group(1) if match else ''


# # ============================================================
# # 📦 LOAD PIPELINE
# # ============================================================

# def load_pipeline(model_dir: str) -> dict:
#     """Load all saved models and encoders from training."""
#     print("=" * 80)
#     print("📦 LOADING SAVED PIPELINE")
#     print("=" * 80)

#     model_section   = pickle.load(open(os.path.join(model_dir, "model_section.pkl"),  "rb"))
#     model_cluster   = pickle.load(open(os.path.join(model_dir, "model_cluster.pkl"),  "rb"))
#     pipeline_config = pickle.load(open(os.path.join(model_dir, "pipeline_config.pkl"), "rb"))

#     # Numeric block lookup table (used to compute rank for new data)
#     df_num_path = os.path.join(model_dir, "df_oiltek_num.pkl")
#     df_lookup   = pickle.load(open(df_num_path, "rb")) if os.path.exists(df_num_path) else None

#     print("   ✅ model_section.pkl     loaded")
#     print("   ✅ model_cluster.pkl     loaded")
#     print("   ✅ pipeline_config.pkl   loaded")
#     print(f"   {'✅' if df_lookup is not None else '⚠ '} df_oiltek_num.pkl       {'loaded' if df_lookup is not None else 'not found — rank will default to 1'}")
#     print(f"\n   Section classes : {list(pipeline_config['le_section'].classes_)}")
#     print(f"   Cluster classes : {list(pipeline_config['le_cluster'].classes_)}")

#     return {
#         "model_section"   : model_section,
#         "model_cluster"   : model_cluster,
#         "config"          : pipeline_config,
#         "df_lookup"       : df_lookup,
#     }


# # ============================================================
# # 🔧 BUILD FEATURES FOR ONE OR MORE DEVICES
# # ============================================================

# def build_features(records: list[dict], config: dict, df_lookup: pd.DataFrame | None) -> pd.DataFrame:
#     """
#     records : list of dicts with keys: device_id, customer, project
#     Returns a DataFrame ready for model inference.
#     """
#     df = pd.DataFrame(records)
#     df.columns = df.columns.str.upper()   # normalise column names

#     le_customer     = config['le_customer']
#     le_project      = config['le_project']
#     le_prefix       = config['le_prefix']
#     le_suffix_lt    = config['le_suffix_lt']
#     le_suffix_last  = config['le_suffix_last']

#     # ── BASIC DEVICE ID FEATURES ────────────────────────────
#     df['numeric_block']          = df['DEVICE_ID'].apply(extract_numeric_block)
#     df['device_prefix']          = df['DEVICE_ID'].apply(extract_prefix)
#     df['device_suffix_letter']   = df['DEVICE_ID'].apply(extract_suffix_letters)
#     df['suffix_full']            = df['DEVICE_ID'].apply(extract_suffix_full)
#     df['prefix_length']          = df['device_prefix'].str.len()
#     df['device_id_length']       = df['DEVICE_ID'].astype(str).str.len()
#     df['has_suffix_letter']      = (df['device_suffix_letter'] != '').astype(int)
#     df['has_numeric']            = (df['numeric_block'] != -1).astype(int)

#     # ── SUFFIX FEATURES ─────────────────────────────────────
#     df['suffix_length']             = df['suffix_full'].astype(str).str.len()
#     df['suffix_has_digit']          = df['suffix_full'].astype(str).str.contains(r'\d', regex=True).astype(int)
#     df['suffix_has_letter']         = df['suffix_full'].astype(str).str.contains(r'[A-Za-z]', regex=True).astype(int)
#     df['suffix_has_decimal']        = df['suffix_full'].astype(str).str.contains(r'\.', regex=True).astype(int)
#     df['suffix_digit_count']        = df['suffix_full'].astype(str).str.count(r'\d')
#     df['suffix_letter_count']       = df['suffix_full'].astype(str).str.count(r'[A-Za-z]')
#     df['suffix_starts_with_digit']  = df['suffix_full'].astype(str).str[0].str.isdigit().fillna(0).astype(int)
#     df['suffix_last_char']          = df['suffix_full'].astype(str).str[-1]
#     df['suffix_last_char_is_letter']= df['suffix_last_char'].str.isalpha().fillna(0).astype(int)
#     df['suffix_last_char_is_digit'] = df['suffix_last_char'].str.isdigit().fillna(0).astype(int)

#     # ── EQUIPMENT ID FEATURES ────────────────────────────────
#     df['equip_id_length']       = df['DEVICE_ID'].astype(str).str.len()
#     df['equip_id_digit_count']  = df['DEVICE_ID'].astype(str).str.count(r'\d')
#     df['equip_id_letter_count'] = df['DEVICE_ID'].astype(str).str.count(r'[A-Za-z]')

#     # ── NUMERIC BLOCK RANK ──────────────────────────────────
#     # Compute rank using training lookup; fall back to 1 if not found
#     def get_block_rank(row):
#         if df_lookup is None:
#             return 1
#         subset = df_lookup[
#             (df_lookup['CUSTOMER'] == row['CUSTOMER']) &
#             (df_lookup['PROJECT']  == row['PROJECT'])
#         ]
#         if subset.empty:
#             return 1
#         unique_blocks = sorted(subset['numeric_block'].unique())
#         block_to_rank = {b: i + 1 for i, b in enumerate(unique_blocks)}
#         return block_to_rank.get(row['numeric_block'], 1)

#     df['numeric_block_rank'] = df.apply(get_block_rank, axis=1)

#     # ── LABEL ENCODING (handle unseen labels gracefully) ────
#     def safe_encode(le, values):
#         known = set(le.classes_)
#         return np.array([
#             le.transform([v])[0] if v in known else 0
#             for v in values
#         ])

#     df['customer_enc']         = safe_encode(le_customer,    df['CUSTOMER'])
#     df['project_enc']          = safe_encode(le_project,     df['PROJECT'])
#     df['prefix_enc']           = safe_encode(le_prefix,      df['device_prefix'])
#     df['suffix_letter_enc']    = safe_encode(le_suffix_lt,   df['device_suffix_letter'])
#     df['suffix_last_char_enc'] = safe_encode(le_suffix_last, df['suffix_last_char'])

#     return df



In [ ]:

# # ============================================================
# # 🚀 PREDICT
# # ============================================================

# def predict(records: list[dict], pipeline: dict, threshold: float = UNKNOWN_THRESHOLD) -> pd.DataFrame:
#     """
#     Runs the chained Section → Cluster prediction.

#     Parameters
#     ----------
#     records   : list of dicts  [{device_id, customer, project}, ...]
#     pipeline  : dict returned by load_pipeline()
#     threshold : float — confidence below this → UNKNOWN

#     Returns
#     -------
#     pd.DataFrame with prediction results and confidence scores.
#     """
#     config        = pipeline["config"]
#     model_section = pipeline["model_section"]
#     model_cluster = pipeline["model_cluster"]
#     df_lookup     = pipeline["df_lookup"]
#     le_section    = config["le_section"]
#     le_cluster    = config["le_cluster"]

#     # ── Build feature matrix ─────────────────────────────────
#     df_feat = build_features(records, config, df_lookup)

#     section_features = config["section_features"]
#     cluster_features = config["cluster_features"]   # includes 'predicted_section'

#     # Ensure only numeric cols used in training are present
#     X_section = df_feat[section_features].apply(pd.to_numeric, errors='coerce').fillna(0)

#     # ── Stage 1: Section prediction ──────────────────────────
#     sec_proba    = model_section.predict_proba(X_section)          # shape (n, n_section_classes)
#     sec_pred_enc = np.argmax(sec_proba, axis=1)
#     sec_conf     = sec_proba.max(axis=1)

#     # ── Chain: inject predicted section as feature ───────────
#     X_cluster = X_section.copy()
#     # Add predicted_section column (same name as training)
#     X_cluster = df_feat[
#         [f for f in cluster_features if f != 'predicted_section']
#     ].apply(pd.to_numeric, errors='coerce').fillna(0)
#     X_cluster["predicted_section"] = sec_pred_enc
#     # Reorder to match training order
#     X_cluster = X_cluster[cluster_features]

#     # ── Stage 2: Cluster prediction ──────────────────────────
#     clu_proba    = model_cluster.predict_proba(X_cluster)
#     clu_pred_enc = np.argmax(clu_proba, axis=1)
#     clu_conf     = clu_proba.max(axis=1)

#     # ── Decode labels ────────────────────────────────────────
#     sec_labels = le_section.inverse_transform(sec_pred_enc)
#     clu_labels = le_cluster.inverse_transform(clu_pred_enc)

#     # ── Apply UNKNOWN threshold ───────────────────────────────
#     sec_final = np.where(sec_conf >= threshold, sec_labels, "UNKNOWN")
#     clu_final = np.where(clu_conf >= threshold, clu_labels, "UNKNOWN")

#     # ── Build result DataFrame ───────────────────────────────
#     result = pd.DataFrame(records)
#     result.columns = result.columns.str.upper()

#     result["PREDICTED_SECTION"]    = sec_final
#     result["SECTION_CONFIDENCE"]   = np.round(sec_conf * 100, 2)
#     result["PREDICTED_CLUSTER"]    = clu_final
#     result["CLUSTER_CONFIDENCE"]   = np.round(clu_conf * 100, 2)

#     return result


# # ============================================================
# # 🖨️  PRETTY PRINT RESULTS
# # ============================================================

# def print_results(result: pd.DataFrame):
#     print("\n" + "=" * 80)
#     print("📊 PREDICTION RESULTS")
#     print("=" * 80)

#     for _, row in result.iterrows():
#         sec_flag = "⚠ UNKNOWN" if row["PREDICTED_SECTION"] == "UNKNOWN" else "✅"
#         clu_flag = "⚠ UNKNOWN" if row["PREDICTED_CLUSTER"] == "UNKNOWN" else "✅"

#         print(f"\n   Device   : {row['DEVICE_ID']}")
#         print(f"   Customer : {row['CUSTOMER']}   |   Project : {row['PROJECT']}")
#         print(f"   {sec_flag} Section : {row['PREDICTED_SECTION']:20s}  (confidence: {row['SECTION_CONFIDENCE']:.1f}%)")
#         print(f"   {clu_flag} Cluster : {row['PREDICTED_CLUSTER']:20s}  (confidence: {row['CLUSTER_CONFIDENCE']:.1f}%)")
#         print(f"   {'-' * 60}")

#     print(f"\n{'=' * 80}")
#     print(f"   Total records predicted : {len(result)}")
#     unknown_sec = (result["PREDICTED_SECTION"] == "UNKNOWN").sum()
#     unknown_clu = (result["PREDICTED_CLUSTER"] == "UNKNOWN").sum()
#     if unknown_sec or unknown_clu:
#         print(f"   ⚠  UNKNOWN Section : {unknown_sec}   |   UNKNOWN Cluster : {unknown_clu}")
#     print(f"   Confidence threshold used : {UNKNOWN_THRESHOLD * 100:.0f}%")
#     print("=" * 80)


# # ============================================================
# # ▶️  MAIN — EXAMPLE APPLICATION
# # ============================================================

# if __name__ == "__main__":

#     # ----------------------------------------------------------
#     # 🔹 EXAMPLE INPUT DATA
#     #    Format: list of dicts with keys: device_id, customer, project
#     # ----------------------------------------------------------
#     test_records = [
#         # ── Single device (your provided example) ──
#         {"device_id": "LL510",    "customer": "OILTEK", "project": "A1706"},

#         # # ── More examples to stress-test the model ──
#         {"device_id": "HT275",    "customer": "OILTEK", "project": "A1706"},
#         # {"device_id": "CPM6311",  "customer": "OILTEK", "project": "A1706"},
#         # {"device_id": "FIT151",   "customer": "OILTEK", "project": "A1706"},
#         # {"device_id": "V29011",   "customer": "OILTEK", "project": "A1706"},
#         # {"device_id": "PT500A",   "customer": "OILTEK", "project": "A1706"},

#         # # ── Edge cases ──────────────────────────────────────
#         {"device_id": "ZZZ9999X", "customer": "OILTEK", "project": "A1706"},   # unknown-looking device
#         {"device_id": "LL510",    "customer": "UNKNOWN_CUSTOMER", "project": "UNKNOWN_PROJECT"},  # unseen customer/project
#     ]

#     # ----------------------------------------------------------
#     # 1. Load pipeline
#     # ----------------------------------------------------------
#     pipeline = load_pipeline(MODEL_DIR)

#     # ----------------------------------------------------------
#     # 2. Predict
#     # ----------------------------------------------------------
#     result_df = predict(test_records, pipeline, threshold=UNKNOWN_THRESHOLD)

#     # ----------------------------------------------------------
#     # 3. Print results
#     # ----------------------------------------------------------
#     print_results(result_df)

#     # ----------------------------------------------------------
#     # 4. Optional: export to Excel
#     # ----------------------------------------------------------
#     out_path = os.path.join(MODEL_DIR_OUTPUT_PREDICTION, "application_results.xlsx")
#     result_df.to_excel(out_path, index=False)
#     print(f"\n   💾 Results saved to: {out_path}")


# # ============================================================
# # 🔁  SINGLE-DEVICE CONVENIENCE FUNCTION
# #     Use this for quick one-off lookups in other scripts
# # ============================================================

# def predict_single(device_id: str, customer: str, project: str,
#                    pipeline: dict, threshold: float = UNKNOWN_THRESHOLD) -> dict:
#     """
#     Shorthand wrapper for predicting a single device.

#     Example
#     -------
#     >>> pipeline = load_pipeline(MODEL_DIR)
#     >>> result = predict_single("LL510", "OILTEK", "A1706", pipeline)
#     >>> print(result)
#     """
#     result = predict(
#         [{"device_id": device_id, "customer": customer, "project": project}],
#         pipeline,
#         threshold=threshold
#     )
#     row = result.iloc[0]
#     return {
#         "device_id"           : row["DEVICE_ID"],
#         "customer"            : row["CUSTOMER"],
#         "project"             : row["PROJECT"],
#         "predicted_section"   : row["PREDICTED_SECTION"],
#         "section_confidence"  : row["SECTION_CONFIDENCE"],
#         "predicted_cluster"   : row["PREDICTED_CLUSTER"],
#         "cluster_confidence"  : row["CLUSTER_CONFIDENCE"],
#     }

# # output is not satisfy. need to focus on how unknown customer still manage to catch up the prediction. it shouldn't supposedly

# EDIT NEW SCRIPT (Wed 22/4)

In [ ]:
# # ============================================================
# # 🤖 DEVICE CLUSTER — MODEL APPLICATION SCRIPT
# # ============================================================
# # Loads saved pipeline (models + encoders) from training phase
# # and predicts SECTION + CLUSTER for new device entries.
# #
# # Guard rules (evaluated BEFORE model inference):
# #   • Unseen CUSTOMER          → BLOCKED (confidence: N/A)
# #   • Unseen PROJECT           → BLOCKED (confidence: N/A)
# #   • Confidence < threshold   → UNKNOWN (shown with actual %)
# # ============================================================

# import pandas as pd
# import numpy as np
# import re
# import pickle
# import os
# import warnings
# warnings.filterwarnings('ignore')


# # ============================================================
# # 📂 PATHS — Update to match your saved model directory
# # ============================================================
# MODEL_DIR = r"C:\Users\sitisyaziyah\source\repos\DeviceCluster\1.training_model\model_config"

# UNKNOWN_THRESHOLD = 0.50   # confidence below this → UNKNOWN


# # ============================================================
# # 🔧 FEATURE ENGINEERING FUNCTIONS (must mirror training)
# # ============================================================

# def extract_numeric_block(device_id):
#     match = re.search(r'\d+', str(device_id))
#     return int(match.group()) if match else -1

# def extract_prefix(device_id):
#     match = re.match(r'^([A-Za-z]+)', str(device_id))
#     return match.group(1).upper() if match else ''

# def extract_suffix_letters(device_id):
#     match = re.search(r'\d+([A-Za-z]*)$', str(device_id))
#     return match.group(1).upper() if match else ''

# def extract_suffix_full(device_id):
#     match = re.search(r'\d+(.*)$', str(device_id))
#     return match.group(1) if match else ''


# # ============================================================
# # 📦 LOAD PIPELINE
# # ============================================================

# def load_pipeline(model_dir: str) -> dict:
#     """Load all saved models and encoders from training."""
#     print("=" * 80)
#     print("📦 LOADING SAVED PIPELINE")
#     print("=" * 80)

#     model_section   = pickle.load(open(os.path.join(model_dir, "model_section.pkl"),  "rb"))
#     model_cluster   = pickle.load(open(os.path.join(model_dir, "model_cluster.pkl"),  "rb"))
#     pipeline_config = pickle.load(open(os.path.join(model_dir, "pipeline_config.pkl"), "rb"))

#     df_num_path = os.path.join(model_dir, "df_oiltek_num.pkl")
#     df_lookup   = pickle.load(open(df_num_path, "rb")) if os.path.exists(df_num_path) else None

#     print("   ✅ model_section.pkl     loaded")
#     print("   ✅ model_cluster.pkl     loaded")
#     print("   ✅ pipeline_config.pkl   loaded")
#     print(f"   {'✅' if df_lookup is not None else '⚠ '} df_oiltek_num.pkl       "
#           f"{'loaded' if df_lookup is not None else 'not found — rank will default to 1'}")
#     print(f"\n   Known customers : {sorted(pipeline_config['le_customer'].classes_)}")
#     print(f"   Known projects  : {sorted(pipeline_config['le_project'].classes_)}")
#     print(f"   Section classes : {list(pipeline_config['le_section'].classes_)}")
#     print(f"   Cluster classes : {list(pipeline_config['le_cluster'].classes_)}")

#     return {
#         "model_section" : model_section,
#         "model_cluster" : model_cluster,
#         "config"        : pipeline_config,
#         "df_lookup"     : df_lookup,
#     }


# # ============================================================
# # 🛡️  GUARD — DETECT UNSEEN ENTITIES BEFORE INFERENCE
# # ============================================================

# def check_unseen(df: pd.DataFrame, config: dict) -> pd.Series:
#     """
#     Returns a Series of rejection reasons per row.
#     Empty string = record is eligible for inference.

#     WHY this matters
#     ----------------
#     LabelEncoder silently maps unseen values to index 0, which
#     resolves to a real known entity and produces confident but
#     completely meaningless predictions.  This guard stops that.
#     """
#     known_customers = set(config['le_customer'].classes_)
#     known_projects  = set(config['le_project'].classes_)

#     reasons = []
#     for _, row in df.iterrows():
#         row_reasons = []
#         if row['CUSTOMER'] not in known_customers:
#             row_reasons.append(f"unseen CUSTOMER '{row['CUSTOMER']}'")
#         if row['PROJECT'] not in known_projects:
#             row_reasons.append(f"unseen PROJECT '{row['PROJECT']}'")
#         reasons.append("; ".join(row_reasons))

#     return pd.Series(reasons, index=df.index)


# # ============================================================
# # 🔧 BUILD FEATURES (eligible rows only)
# # ============================================================

# def build_features(df: pd.DataFrame, config: dict, df_lookup) -> pd.DataFrame:
#     """
#     df must already be guard-checked (all customers/projects known).
#     prefix/suffix letters may still be unseen → safely fall back to 0.
#     """
#     le_customer    = config['le_customer']
#     le_project     = config['le_project']
#     le_prefix      = config['le_prefix']
#     le_suffix_lt   = config['le_suffix_lt']
#     le_suffix_last = config['le_suffix_last']

#     df = df.copy()

#     # ── BASIC DEVICE ID FEATURES ────────────────────────────
#     df['numeric_block']         = df['DEVICE_ID'].apply(extract_numeric_block)
#     df['device_prefix']         = df['DEVICE_ID'].apply(extract_prefix)
#     df['device_suffix_letter']  = df['DEVICE_ID'].apply(extract_suffix_letters)
#     df['suffix_full']           = df['DEVICE_ID'].apply(extract_suffix_full)
#     df['prefix_length']         = df['device_prefix'].str.len()
#     df['device_id_length']      = df['DEVICE_ID'].astype(str).str.len()
#     df['has_suffix_letter']     = (df['device_suffix_letter'] != '').astype(int)
#     df['has_numeric']           = (df['numeric_block'] != -1).astype(int)

#     # ── SUFFIX FEATURES ─────────────────────────────────────
#     df['suffix_length']             = df['suffix_full'].astype(str).str.len()
#     df['suffix_has_digit']          = df['suffix_full'].astype(str).str.contains(r'\d', regex=True).astype(int)
#     df['suffix_has_letter']         = df['suffix_full'].astype(str).str.contains(r'[A-Za-z]', regex=True).astype(int)
#     df['suffix_has_decimal']        = df['suffix_full'].astype(str).str.contains(r'\.', regex=True).astype(int)
#     df['suffix_digit_count']        = df['suffix_full'].astype(str).str.count(r'\d')
#     df['suffix_letter_count']       = df['suffix_full'].astype(str).str.count(r'[A-Za-z]')
#     df['suffix_starts_with_digit']  = df['suffix_full'].astype(str).str[0].str.isdigit().fillna(0).astype(int)
#     df['suffix_last_char']          = df['suffix_full'].astype(str).str[-1]
#     df['suffix_last_char_is_letter']= df['suffix_last_char'].str.isalpha().fillna(0).astype(int)
#     df['suffix_last_char_is_digit'] = df['suffix_last_char'].str.isdigit().fillna(0).astype(int)

#     # ── EQUIPMENT ID FEATURES ────────────────────────────────
#     df['equip_id_length']       = df['DEVICE_ID'].astype(str).str.len()
#     df['equip_id_digit_count']  = df['DEVICE_ID'].astype(str).str.count(r'\d')
#     df['equip_id_letter_count'] = df['DEVICE_ID'].astype(str).str.count(r'[A-Za-z]')

#     # ── NUMERIC BLOCK RANK ───────────────────────────────────
#     def get_block_rank(row):
#         if df_lookup is None:
#             return 1
#         subset = df_lookup[
#             (df_lookup['CUSTOMER'] == row['CUSTOMER']) &
#             (df_lookup['PROJECT']  == row['PROJECT'])
#         ]
#         if subset.empty:
#             return 1
#         unique_blocks = sorted(subset['numeric_block'].unique())
#         block_to_rank = {b: i + 1 for i, b in enumerate(unique_blocks)}
#         return block_to_rank.get(row['numeric_block'], 1)

#     df['numeric_block_rank'] = df.apply(get_block_rank, axis=1)

#     # ── LABEL ENCODING ──────────────────────────────────────
#     # customer/project: guaranteed known at this point → direct transform
#     # prefix/suffix: may be unseen → safe fallback to 0
#     def safe_encode(le, values):
#         known = set(le.classes_)
#         return np.array([
#             le.transform([v])[0] if v in known else 0
#             for v in values
#         ])

#     df['customer_enc']         = le_customer.transform(df['CUSTOMER'])
#     df['project_enc']          = le_project.transform(df['PROJECT'])
#     df['prefix_enc']           = safe_encode(le_prefix,      df['device_prefix'])
#     df['suffix_letter_enc']    = safe_encode(le_suffix_lt,   df['device_suffix_letter'])
#     df['suffix_last_char_enc'] = safe_encode(le_suffix_last, df['suffix_last_char'])

#     return df


# # ============================================================
# # 🚀 PREDICT
# # ============================================================

# def predict(records: list[dict], pipeline: dict,
#             threshold: float = UNKNOWN_THRESHOLD) -> pd.DataFrame:
#     """
#     Guard check first → model inference only for eligible rows.

#     Returns
#     -------
#     pd.DataFrame with columns:
#         DEVICE_ID, CUSTOMER, PROJECT,
#         PREDICTED_SECTION, SECTION_CONFIDENCE (%),
#         PREDICTED_CLUSTER, CLUSTER_CONFIDENCE (%),
#         REJECTION_REASON   (empty string if eligible)
#     """
#     config        = pipeline["config"]
#     model_section = pipeline["model_section"]
#     model_cluster = pipeline["model_cluster"]
#     df_lookup     = pipeline["df_lookup"]
#     le_section    = config["le_section"]
#     le_cluster    = config["le_cluster"]

#     # ── Normalise input ──────────────────────────────────────
#     base_df = pd.DataFrame(records)
#     base_df.columns = base_df.columns.str.upper()

#     # ── Guard ────────────────────────────────────────────────
#     rejection_reasons = check_unseen(base_df, config)
#     eligible_mask     = rejection_reasons == ""
#     blocked_mask      = ~eligible_mask

#     # ── Pre-fill all rows as UNKNOWN / N/A ───────────────────
#     pred_section = ["UNKNOWN"] * len(base_df)
#     sec_conf     = [None]      * len(base_df)   # None = not evaluated
#     pred_cluster = ["UNKNOWN"] * len(base_df)
#     clu_conf     = [None]      * len(base_df)

#     # ── Run inference on eligible rows only ──────────────────
#     if eligible_mask.any():
#         elig_idx  = base_df.index[eligible_mask].tolist()
#         df_elig   = base_df.loc[elig_idx].copy().reset_index(drop=True)
#         df_feat   = build_features(df_elig, config, df_lookup)

#         section_features = config["section_features"]
#         cluster_features = config["cluster_features"]

#         X_sec = df_feat[section_features].apply(pd.to_numeric, errors='coerce').fillna(0)

#         # Stage 1 — Section
#         sec_proba_raw = model_section.predict_proba(X_sec)
#         sec_pred_enc  = np.argmax(sec_proba_raw, axis=1)
#         sec_conf_raw  = sec_proba_raw.max(axis=1)

#         # Chain
#         X_clu = df_feat[
#             [f for f in cluster_features if f != 'predicted_section']
#         ].apply(pd.to_numeric, errors='coerce').fillna(0).copy()
#         X_clu["predicted_section"] = sec_pred_enc
#         X_clu = X_clu[cluster_features]

#         # Stage 2 — Cluster
#         clu_proba_raw = model_cluster.predict_proba(X_clu)
#         clu_pred_enc  = np.argmax(clu_proba_raw, axis=1)
#         clu_conf_raw  = clu_proba_raw.max(axis=1)

#         # Decode
#         sec_labels = le_section.inverse_transform(sec_pred_enc)
#         clu_labels = le_cluster.inverse_transform(clu_pred_enc)

#         # Apply threshold
#         sec_final = np.where(sec_conf_raw >= threshold, sec_labels, "UNKNOWN")
#         clu_final = np.where(clu_conf_raw >= threshold, clu_labels, "UNKNOWN")

#         # Write back to original positions
#         for i, orig_idx in enumerate(elig_idx):
#             pred_section[orig_idx] = sec_final[i]
#             sec_conf[orig_idx]     = round(float(sec_conf_raw[i]) * 100, 2)
#             pred_cluster[orig_idx] = clu_final[i]
#             clu_conf[orig_idx]     = round(float(clu_conf_raw[i]) * 100, 2)

#     # ── Assemble result ──────────────────────────────────────
#     result = base_df[["DEVICE_ID", "CUSTOMER", "PROJECT"]].copy()
#     result["PREDICTED_SECTION"]  = pred_section
#     result["SECTION_CONFIDENCE"] = sec_conf       # None for blocked rows
#     result["PREDICTED_CLUSTER"]  = pred_cluster
#     result["CLUSTER_CONFIDENCE"] = clu_conf       # None for blocked rows
#     result["REJECTION_REASON"]   = rejection_reasons.values

#     return result


# # ============================================================
# # 🖨️  PRETTY PRINT RESULTS
# # ============================================================

# def print_results(result: pd.DataFrame):
#     print("\n" + "=" * 80)
#     print("📊 PREDICTION RESULTS")
#     print("=" * 80)

#     for _, row in result.iterrows():
#         rejected = bool(row["REJECTION_REASON"])
#         print(f"\n   Device   : {row['DEVICE_ID']}")
#         print(f"   Customer : {row['CUSTOMER']}   |   Project : {row['PROJECT']}")

#         if rejected:
#             print(f"   🚫 BLOCKED  — {row['REJECTION_REASON']}")
#             print(f"   ⛔ Section : {'UNKNOWN':20s}  (confidence: N/A — not evaluated)")
#             print(f"   ⛔ Cluster : {'UNKNOWN':20s}  (confidence: N/A — not evaluated)")
#         else:
#             sec_unk  = row["PREDICTED_SECTION"] == "UNKNOWN"
#             clu_unk  = row["PREDICTED_CLUSTER"] == "UNKNOWN"
#             sec_icon = "⚠" if sec_unk else "✅"
#             clu_icon = "⚠" if clu_unk else "✅"
#             sec_conf_str = f"{row['SECTION_CONFIDENCE']:.1f}%" if row["SECTION_CONFIDENCE"] is not None else "N/A"
#             clu_conf_str = f"{row['CLUSTER_CONFIDENCE']:.1f}%" if row["CLUSTER_CONFIDENCE"] is not None else "N/A"

#             print(f"   {sec_icon} Section : {row['PREDICTED_SECTION']:20s}  (confidence: {sec_conf_str})")
#             print(f"   {clu_icon} Cluster : {row['PREDICTED_CLUSTER']:20s}  (confidence: {clu_conf_str})")

#         print(f"   {'-' * 60}")

#     # ── Summary ──────────────────────────────────────────────
#     total     = len(result)
#     blocked   = (result["REJECTION_REASON"] != "").sum()
#     evaluated = total - blocked
#     unk_sec   = ((result["PREDICTED_SECTION"] == "UNKNOWN") & (result["REJECTION_REASON"] == "")).sum()
#     unk_clu   = ((result["PREDICTED_CLUSTER"] == "UNKNOWN") & (result["REJECTION_REASON"] == "")).sum()

#     print(f"\n{'=' * 80}")
#     print(f"   Total records              : {total}")
#     print(f"   Evaluated by model         : {evaluated}   |   Blocked (unseen entity) : {blocked}")
#     if unk_sec or unk_clu:
#         print(f"   Low-confidence UNKNOWN  →  Section : {unk_sec}   |   Cluster : {unk_clu}")
#     print(f"   Confidence threshold used  : {UNKNOWN_THRESHOLD * 100:.0f}%")
#     print("=" * 80)


# # ============================================================
# # ▶️  MAIN — EXAMPLE APPLICATION
# # ============================================================

# if __name__ == "__main__":

#     test_records = [
#         # ── Your primary example ─────────────────────────────
#         {"device_id": "LL510",    "customer": "OILTEK", "project": "A1706"},

#         # ── From your notebook error analysis table ──────────
#         {"device_id": "HT275",    "customer": "OILTEK", "project": "A1706"},
#         {"device_id": "CPM6311",  "customer": "OILTEK", "project": "A1706"},
#         {"device_id": "FIT151",   "customer": "OILTEK", "project": "A1706"},
#         {"device_id": "V29011",   "customer": "OILTEK", "project": "A1706"},
#         {"device_id": "PT500A",   "customer": "OILTEK", "project": "A1706"},

#         # ── Edge: unusual device, known customer/project ─────
#         {"device_id": "ZZZ9999X", "customer": "OILTEK",           "project": "A1706"},

#         # ── Edge: unseen customer only → BLOCKED ─────────────
#         {"device_id": "LL510",    "customer": "UNKNOWN_CUSTOMER", "project": "A1706"},

#         # ── Edge: unseen project only → BLOCKED ──────────────
#         {"device_id": "LL510",    "customer": "OILTEK",           "project": "UNKNOWN_PROJECT"},

#         # ── Edge: both unseen → BLOCKED with combined reason ─
#         {"device_id": "LL510",    "customer": "UNKNOWN_CUSTOMER", "project": "UNKNOWN_PROJECT"},
#     ]

#     # 1. Load
#     pipeline = load_pipeline(MODEL_DIR)

#     # 2. Predict
#     result_df = predict(test_records, pipeline, threshold=UNKNOWN_THRESHOLD)

#     # 3. Print
#     print_results(result_df)

#     # 4. Export
#     out_path = os.path.join(MODEL_DIR, "application_results.xlsx")
#     result_df.to_excel(out_path, index=False)
#     print(f"\n   💾 Results saved to: {out_path}")


# # ============================================================
# # 🔁  SINGLE-DEVICE CONVENIENCE FUNCTION
# # ============================================================

# def predict_single(device_id: str, customer: str, project: str,
#                    pipeline: dict, threshold: float = UNKNOWN_THRESHOLD) -> dict:
#     """
#     Shorthand for one device. Returns a plain dict.

#     Example
#     -------
#     >>> pipeline = load_pipeline(MODEL_DIR)
#     >>> result = predict_single("LL510", "OILTEK", "A1706", pipeline)
#     >>> print(result)
#     """
#     result = predict(
#         [{"device_id": device_id, "customer": customer, "project": project}],
#         pipeline,
#         threshold=threshold,
#     )
#     row = result.iloc[0]
#     return {
#         "device_id"          : row["DEVICE_ID"],
#         "customer"           : row["CUSTOMER"],
#         "project"            : row["PROJECT"],
#         "predicted_section"  : row["PREDICTED_SECTION"],
#         "section_confidence" : row["SECTION_CONFIDENCE"],   # None if blocked
#         "predicted_cluster"  : row["PREDICTED_CLUSTER"],
#         "cluster_confidence" : row["CLUSTER_CONFIDENCE"],   # None if blocked
#         "rejection_reason"   : row["REJECTION_REASON"],
#     }

# SCRIPT (Wed 23/4)

How does this script and model behave? <br>
Hanging scirpt since claude couldnt load


In [11]:
# ============================================================
# 🤖 DEVICE CLUSTER — MODEL APPLICATION SCRIPT
# ============================================================
# Loads saved pipeline (models + encoders) from training phase
# and predicts SECTION + CLUSTER for new device entries.
#
# Guard rules (evaluated BEFORE model inference):
#   • Unseen CUSTOMER          → BLOCKED (confidence: N/A)
#   • Unseen PROJECT           → BLOCKED (confidence: N/A)
#   • Confidence < threshold   → UNKNOWN (shown with actual %)
#
# IMPORTANT: SafeLabelEncoder must be defined here so that
# pickle.load() can deserialise the saved pipeline_config.pkl.
# The class definition must exactly match the one used during
# training — do not modify it independently.
# ============================================================

import pandas as pd
import numpy as np
import re
import pickle
import os
import warnings
from sklearn.preprocessing import LabelEncoder
warnings.filterwarnings('ignore')


# ============================================================
# 📂 PATHS
# ============================================================
MODEL_DIR = r"C:\Users\sitisyaziyah\source\repos\DeviceCluster\1.training_model\model_config"

UNKNOWN_THRESHOLD = 0.50   # confidence below this → UNKNOWN


# ============================================================
# 🔐 SafeLabelEncoder
#    Must be defined before pickle.load() is called.
#    This definition must stay identical to the one in the
#    training script — pickle restores the class by reference.
# ============================================================

class SafeLabelEncoder:
    """
    LabelEncoder that handles unseen labels gracefully.
    Adds '__UNKNOWN__' as a sentinel class during fit.
    Unseen values during transform map to '__UNKNOWN__'
    instead of raising a ValueError.
    """

    UNKNOWN_LABEL = "__UNKNOWN__"

    def __init__(self):
        self._le = LabelEncoder()
        self.classes_ = None

    def fit(self, y):
        labels = list(pd.Series(y).astype(str).unique())
        if self.UNKNOWN_LABEL not in labels:
            labels = [self.UNKNOWN_LABEL] + labels
        self._le.fit(labels)
        self.classes_ = self._le.classes_
        return self

    def transform(self, y):
        y_str = pd.Series(y).astype(str)
        known = set(self.classes_)
        y_safe = y_str.where(y_str.isin(known), other=self.UNKNOWN_LABEL)
        return self._le.transform(y_safe)

    def fit_transform(self, y):
        return self.fit(y).transform(y)

    def inverse_transform(self, y):
        return self._le.inverse_transform(y)

    def is_known(self, values):
        known = set(self.classes_) - {self.UNKNOWN_LABEL}
        return pd.Series(values).astype(str).isin(known)

    def real_classes(self):
        """Return classes_ excluding the __UNKNOWN__ sentinel."""
        return [c for c in self.classes_ if c != self.UNKNOWN_LABEL]


# ============================================================
# 🔧 FEATURE ENGINEERING FUNCTIONS (mirrors training exactly)
# ============================================================

def extract_numeric_block(device_id):
    match = re.search(r'\d+', str(device_id))
    return int(match.group()) if match else -1

def extract_prefix(device_id):
    match = re.match(r'^([A-Za-z]+)', str(device_id))
    return match.group(1).upper() if match else ''

def extract_suffix_letters(device_id):
    match = re.search(r'\d+([A-Za-z]*)$', str(device_id))
    return match.group(1).upper() if match else ''

def extract_suffix_full(device_id):
    match = re.search(r'\d+(.*)$', str(device_id))
    return match.group(1) if match else ''


# ============================================================
# 📦 LOAD PIPELINE
# ============================================================

def load_pipeline(model_dir: str) -> dict:
    """
    Load saved models and encoders from the training phase.

    SafeLabelEncoder must be defined above before this is
    called — pickle needs the class in scope to deserialise.
    """
    print("=" * 80)
    print("📦 LOADING SAVED PIPELINE")
    print("=" * 80)

    model_section   = pickle.load(open(os.path.join(model_dir, "model_section.pkl"),  "rb"))
    model_cluster   = pickle.load(open(os.path.join(model_dir, "model_cluster.pkl"),  "rb"))
    pipeline_config = pickle.load(open(os.path.join(model_dir, "pipeline_config.pkl"), "rb"))

    df_num_path = os.path.join(model_dir, "df_oiltek_num.pkl")
    df_lookup   = pickle.load(open(df_num_path, "rb")) if os.path.exists(df_num_path) else None

    # ── known_customers comes directly from pipeline_config ──
    # It was saved during training as a plain Python set of
    # customer strings — no __UNKNOWN__ included.
    # Do NOT rebuild it from le_customer.classes_ here, because
    # classes_ contains __UNKNOWN__ and would pollute the gate.
    known_customers = pipeline_config['known_customers']
    known_projects  = set(pipeline_config['le_project'].real_classes())

    print("   ✅ model_section.pkl     loaded")
    print("   ✅ model_cluster.pkl     loaded")
    print("   ✅ pipeline_config.pkl   loaded")
    print(f"   {'✅' if df_lookup is not None else '⚠ '} df_oiltek_num.pkl       "
          f"{'loaded' if df_lookup is not None else 'not found — rank will default to 1'}")
    print(f"\n   Known customers : {sorted(known_customers)}")
    print(f"   Known projects  : {sorted(known_projects)}")
    print(f"   Section classes : {pipeline_config['le_section'].real_classes()}")
    print(f"   Cluster classes : {pipeline_config['le_cluster'].real_classes()}")

    return {
        "model_section"   : model_section,
        "model_cluster"   : model_cluster,
        "config"          : pipeline_config,
        "df_lookup"       : df_lookup,
        "known_customers" : known_customers,
        "known_projects"  : known_projects,
    }


# ============================================================
# 🛡️  GUARD — DETECT UNSEEN ENTITIES BEFORE INFERENCE
# ============================================================

def check_unseen(df: pd.DataFrame, pipeline: dict) -> pd.Series:
    """
    Returns a Series of rejection reasons per row.
    Empty string = row is eligible for inference.

    Uses known_customers and known_projects from the pipeline
    dict (saved during training) — NOT from le.classes_ directly,
    because classes_ includes the __UNKNOWN__ sentinel.
    """
    known_customers = pipeline["known_customers"]
    known_projects  = pipeline["known_projects"]

    reasons = []
    for _, row in df.iterrows():
        row_reasons = []
        if str(row['CUSTOMER']) not in known_customers:
            row_reasons.append(f"unseen CUSTOMER '{row['CUSTOMER']}'")
        if str(row['PROJECT']) not in known_projects:
            row_reasons.append(f"unseen PROJECT '{row['PROJECT']}'")
        reasons.append("; ".join(row_reasons))

    return pd.Series(reasons, index=df.index)


# ============================================================
# 🔧 BUILD FEATURES (eligible rows only)
# ============================================================

def build_features(df: pd.DataFrame, config: dict, df_lookup) -> pd.DataFrame:
    """
    Constructs the full feature matrix for eligible rows.
    Mirrors the feature engineering in the training script exactly.

    customer/project: guaranteed known at this point (guard passed).
    prefix/suffix:    may be unseen → SafeLabelEncoder maps to
                      __UNKNOWN__ index automatically, no crash.
    """
    le_customer    = config['le_customer']
    le_project     = config['le_project']
    le_prefix      = config['le_prefix']
    le_suffix_lt   = config['le_suffix_lt']       # key name matches training save
    le_suffix_last = config['le_suffix_last']

    df = df.copy()

    # ── Basic device ID features ─────────────────────────────
    df['numeric_block']         = df['DEVICE_ID'].apply(extract_numeric_block)
    df['device_prefix']         = df['DEVICE_ID'].apply(extract_prefix)
    df['device_suffix_letter']  = df['DEVICE_ID'].apply(extract_suffix_letters)
    df['suffix_full']           = df['DEVICE_ID'].apply(extract_suffix_full)
    df['prefix_length']         = df['device_prefix'].str.len()
    df['device_id_length']      = df['DEVICE_ID'].astype(str).str.len()
    df['has_suffix_letter']     = (df['device_suffix_letter'] != '').astype(int)
    df['has_numeric']           = (df['numeric_block'] != -1).astype(int)

    # ── Advanced suffix features ─────────────────────────────
    df['suffix_length']             = df['suffix_full'].astype(str).str.len()
    df['suffix_has_digit']          = df['suffix_full'].astype(str).str.contains(r'\d', regex=True).astype(int)
    df['suffix_has_letter']         = df['suffix_full'].astype(str).str.contains(r'[A-Za-z]', regex=True).astype(int)
    df['suffix_has_decimal']        = df['suffix_full'].astype(str).str.contains(r'\.', regex=True).astype(int)
    df['suffix_digit_count']        = df['suffix_full'].astype(str).str.count(r'\d')
    df['suffix_letter_count']       = df['suffix_full'].astype(str).str.count(r'[A-Za-z]')
    df['suffix_starts_with_digit']  = df['suffix_full'].astype(str).str[0].apply(lambda c: 1 if isinstance(c, str) and c.isdigit() else 0)
    df['suffix_last_char']          = df['suffix_full'].astype(str).str[-1].fillna('')
    df['suffix_last_char_is_letter']= df['suffix_last_char'].apply(lambda c: 1 if isinstance(c, str) and c.isalpha() else 0)
    df['suffix_last_char_is_digit'] = df['suffix_last_char'].apply(lambda c: 1 if isinstance(c, str) and c.isdigit() else 0)

    # ── Equipment ID features ─────────────────────────────────
    df['equip_id_length']       = df['DEVICE_ID'].astype(str).str.len()
    df['equip_id_digit_count']  = df['DEVICE_ID'].astype(str).str.count(r'\d')
    df['equip_id_letter_count'] = df['DEVICE_ID'].astype(str).str.count(r'[A-Za-z]')

    # ── Numeric block rank ────────────────────────────────────
    # Looks up the rank from training data via df_oiltek_num.pkl.
    # Falls back to 1 if the lookup table is missing or the
    # customer/project/block combination is not found.
    def get_block_rank(row):
        if df_lookup is None:
            return 1
        subset = df_lookup[
            (df_lookup['CUSTOMER'] == row['CUSTOMER']) &
            (df_lookup['PROJECT']  == row['PROJECT'])
        ]
        if subset.empty:
            return 1
        unique_blocks = sorted(subset['numeric_block'].unique())
        block_to_rank = {b: i + 1 for i, b in enumerate(unique_blocks)}
        return block_to_rank.get(row['numeric_block'], 1)

    df['numeric_block_rank'] = df.apply(get_block_rank, axis=1)

    # ── Label encoding ────────────────────────────────────────
    # SafeLabelEncoder.transform() handles unseen values by
    # mapping them to __UNKNOWN__ — no manual safe_encode()
    # wrapper needed.
    df['customer_enc']         = le_customer.transform(df['CUSTOMER'])
    df['project_enc']          = le_project.transform(df['PROJECT'])
    df['prefix_enc']           = le_prefix.transform(df['device_prefix'])
    df['suffix_letter_enc']    = le_suffix_lt.transform(df['device_suffix_letter'])
    df['suffix_last_char_enc'] = le_suffix_last.transform(df['suffix_last_char'])

    return df


# ============================================================
# 🚀 PREDICT
# ============================================================

def predict(records: list[dict], pipeline: dict,
            threshold: float = UNKNOWN_THRESHOLD) -> pd.DataFrame:
    """
    Guard check → feature engineering → chained model inference.

    Flow
    ----
    1. Normalise column names to uppercase.
    2. check_unseen() flags rows with unknown customer/project.
    3. Blocked rows get UNKNOWN / N/A immediately.
    4. Eligible rows go through build_features() then the
       two-stage section → cluster chain.
    5. Predictions below `threshold` are replaced with UNKNOWN.

    Returns
    -------
    pd.DataFrame with columns:
        DEVICE_ID, CUSTOMER, PROJECT,
        PREDICTED_SECTION, SECTION_CONFIDENCE (%),
        PREDICTED_CLUSTER, CLUSTER_CONFIDENCE (%),
        REJECTION_REASON   (empty string if eligible)
    """
    config        = pipeline["config"]
    model_section = pipeline["model_section"]
    model_cluster = pipeline["model_cluster"]
    df_lookup     = pipeline["df_lookup"]
    le_section    = config["le_section"]
    le_cluster    = config["le_cluster"]

    section_features = config["section_features"]   # X_train columns (no predicted_section)
    cluster_features = config["cluster_features"]   # X_train_chained columns (with predicted_section)

    # ── Normalise input ───────────────────────────────────────
    base_df = pd.DataFrame(records)
    base_df.columns = base_df.columns.str.upper()

    # ── Guard ─────────────────────────────────────────────────
    rejection_reasons = check_unseen(base_df, pipeline)
    eligible_mask     = rejection_reasons == ""
    blocked_mask      = ~eligible_mask

    # ── Pre-fill all rows as UNKNOWN / N/A ────────────────────
    pred_section = ["UNKNOWN"] * len(base_df)
    sec_conf     = [None]      * len(base_df)    # None = not evaluated (blocked)
    pred_cluster = ["UNKNOWN"] * len(base_df)
    clu_conf     = [None]      * len(base_df)

    # ── Run inference on eligible rows only ───────────────────
    if eligible_mask.any():
        elig_idx = base_df.index[eligible_mask].tolist()
        df_elig  = base_df.loc[elig_idx].copy().reset_index(drop=True)
        df_feat  = build_features(df_elig, config, df_lookup)

        X_sec = (
            df_feat[section_features]
            .apply(pd.to_numeric, errors='coerce')
            .fillna(0)
        )

        # Stage 1 — Section
        sec_proba_raw = model_section.predict_proba(X_sec)
        sec_pred_enc  = np.argmax(sec_proba_raw, axis=1)
        sec_conf_raw  = sec_proba_raw.max(axis=1)

        # Decode section — map __UNKNOWN__ index back to "UNKNOWN" string
        sec_decoded = le_section.inverse_transform(sec_pred_enc)
        sec_decoded = np.where(
            sec_decoded == SafeLabelEncoder.UNKNOWN_LABEL,
            "UNKNOWN",
            sec_decoded
        )

        # Chain — build cluster feature matrix
        X_clu = (
            df_feat[[f for f in cluster_features if f != "predicted_section"]]
            .apply(pd.to_numeric, errors='coerce')
            .fillna(0)
            .copy()
        )
        X_clu["predicted_section"] = sec_pred_enc
        X_clu = X_clu[cluster_features]   # enforce exact column order from training

        # Stage 2 — Cluster
        clu_proba_raw = model_cluster.predict_proba(X_clu)
        clu_pred_enc  = np.argmax(clu_proba_raw, axis=1)
        clu_conf_raw  = clu_proba_raw.max(axis=1)

        # Decode cluster
        clu_decoded = le_cluster.inverse_transform(clu_pred_enc)
        clu_decoded = np.where(
            clu_decoded == SafeLabelEncoder.UNKNOWN_LABEL,
            "UNKNOWN",
            clu_decoded
        )

        # Apply confidence threshold
        sec_final = np.where(sec_conf_raw >= threshold, sec_decoded, "UNKNOWN")
        clu_final = np.where(clu_conf_raw >= threshold, clu_decoded, "UNKNOWN")

        # Write back to original row positions
        for i, orig_idx in enumerate(elig_idx):
            pred_section[orig_idx] = sec_final[i]
            sec_conf[orig_idx]     = round(float(sec_conf_raw[i]) * 100, 2)
            pred_cluster[orig_idx] = clu_final[i]
            clu_conf[orig_idx]     = round(float(clu_conf_raw[i]) * 100, 2)

    # ── Assemble result ───────────────────────────────────────
    result = base_df[["DEVICE_ID", "CUSTOMER", "PROJECT"]].copy()
    result["PREDICTED_SECTION"]  = pred_section
    result["SECTION_CONFIDENCE"] = sec_conf       # None for blocked rows
    result["PREDICTED_CLUSTER"]  = pred_cluster
    result["CLUSTER_CONFIDENCE"] = clu_conf       # None for blocked rows
    result["REJECTION_REASON"]   = rejection_reasons.values

    return result


# ============================================================
# 🖨️  PRETTY PRINT RESULTS
# ============================================================

def print_results(result: pd.DataFrame):
    print("\n" + "=" * 80)
    print("📊 PREDICTION RESULTS")
    print("=" * 80)

    for _, row in result.iterrows():
        rejected = bool(row["REJECTION_REASON"])
        print(f"\n   Device   : {row['DEVICE_ID']}")
        print(f"   Customer : {row['CUSTOMER']}   |   Project : {row['PROJECT']}")

        if rejected:
            print(f"   🚫 BLOCKED  — {row['REJECTION_REASON']}")
            print(f"   ⛔ Section : {'UNKNOWN':20s}  (confidence: N/A — not evaluated)")
            print(f"   ⛔ Cluster : {'UNKNOWN':20s}  (confidence: N/A — not evaluated)")
        else:
            sec_unk      = row["PREDICTED_SECTION"] == "UNKNOWN"
            clu_unk      = row["PREDICTED_CLUSTER"] == "UNKNOWN"
            sec_icon     = "⚠" if sec_unk else "✅"
            clu_icon     = "⚠" if clu_unk else "✅"
            sec_conf_str = f"{row['SECTION_CONFIDENCE']:.1f}%" if row["SECTION_CONFIDENCE"] is not None else "N/A"
            clu_conf_str = f"{row['CLUSTER_CONFIDENCE']:.1f}%" if row["CLUSTER_CONFIDENCE"] is not None else "N/A"

            print(f"   {sec_icon} Section : {row['PREDICTED_SECTION']:20s}  (confidence: {sec_conf_str})")
            print(f"   {clu_icon} Cluster : {row['PREDICTED_CLUSTER']:20s}  (confidence: {clu_conf_str})")

        print(f"   {'-' * 60}")

    # ── Summary ───────────────────────────────────────────────
    total     = len(result)
    blocked   = (result["REJECTION_REASON"] != "").sum()
    evaluated = total - blocked
    unk_sec   = ((result["PREDICTED_SECTION"] == "UNKNOWN") & (result["REJECTION_REASON"] == "")).sum()
    unk_clu   = ((result["PREDICTED_CLUSTER"] == "UNKNOWN") & (result["REJECTION_REASON"] == "")).sum()

    print(f"\n{'=' * 80}")
    print(f"   Total records              : {total}")
    print(f"   Evaluated by model         : {evaluated}   |   Blocked (unseen entity) : {blocked}")
    if unk_sec or unk_clu:
        print(f"   Low-confidence UNKNOWN  →  Section : {unk_sec}   |   Cluster : {unk_clu}")
    print(f"   Confidence threshold used  : {UNKNOWN_THRESHOLD * 100:.0f}%")
    print("=" * 80)


# ============================================================
# ▶️  MAIN — EXAMPLE APPLICATION
# ============================================================


In [13]:

if __name__ == "__main__":

    test_records = [
        # ── Primary example ──────────────────────────────────
        {"device_id": "LL510",    "customer": "OILTEK", "project": "A1777"}
    ]

    # 1. Load pipeline (SafeLabelEncoder must be defined above)
    pipeline = load_pipeline(MODEL_DIR)

    # 2. Predict
    result_df = predict(test_records, pipeline, threshold=UNKNOWN_THRESHOLD)

    # 3. Print
    print_results(result_df)

    # 4. Export
    out_path = os.path.join(MODEL_DIR, "application_results.xlsx")
    result_df.to_excel(out_path, index=False)
    print(f"\n   💾 Results saved to: {out_path}")



📦 LOADING SAVED PIPELINE
   ✅ model_section.pkl     loaded
   ✅ model_cluster.pkl     loaded
   ✅ pipeline_config.pkl   loaded
   ✅ df_oiltek_num.pkl       loaded

   Known customers : [np.str_('OILTEK'), np.str_('UGS')]
   Known projects  : [np.str_('A1706'), np.str_('A1993'), np.str_('A1994'), np.str_('A1995'), np.str_('A2027'), np.str_('A8881'), np.str_('A8882'), np.str_('A8883'), np.str_('A8884'), np.str_('A8885'), np.str_('A9991'), np.str_('A9992')]
   Section classes : [np.str_('SECTION 1'), np.str_('SECTION 2'), np.str_('SECTION 3'), np.str_('SECTION 4'), np.str_('SECTION 5'), np.str_('SECTION 6')]
   Cluster classes : [np.str_('CLUSTER 1'), np.str_('CLUSTER 2'), np.str_('CLUSTER 3'), np.str_('CLUSTER 4'), np.str_('CLUSTER 5'), np.str_('CLUSTER 6'), np.str_('CLUSTER 7'), np.str_('CLUSTER 8'), np.str_('CLUSTER 9')]

📊 PREDICTION RESULTS

   Device   : LL510
   Customer : OILTEK   |   Project : A1777
   🚫 BLOCKED  — unseen PROJECT 'A1777'
   ⛔ Section : UNKNOWN               (conf

In [ ]:

# ============================================================
# 🔁  SINGLE-DEVICE CONVENIENCE FUNCTION
# ============================================================

def predict_single(device_id: str, customer: str, project: str,
                   pipeline: dict, threshold: float = UNKNOWN_THRESHOLD) -> dict:
    """
    Shorthand for predicting one device at a time.

    Example
    -------
    >>> pipeline = load_pipeline(MODEL_DIR)
    >>> result = predict_single("LL510", "OILTEK", "A1706", pipeline)
    >>> print(result)
    """
    result = predict(
        [{"device_id": device_id, "customer": customer, "project": project}],
        pipeline,
        threshold=threshold,
    )
    row = result.iloc[0]
    return {
        "device_id"          : row["DEVICE_ID"],
        "customer"           : row["CUSTOMER"],
        "project"            : row["PROJECT"],
        "predicted_section"  : row["PREDICTED_SECTION"],
        "section_confidence" : row["SECTION_CONFIDENCE"],   # None if blocked
        "predicted_cluster"  : row["PREDICTED_CLUSTER"],
        "cluster_confidence" : row["CLUSTER_CONFIDENCE"],   # None if blocked
        "rejection_reason"   : row["REJECTION_REASON"],
    }

# edited script [23/4,  16;00]

In [11]:
# ============================================================
# 🤖 DEVICE CLUSTER — MODEL APPLICATION SCRIPT
# ============================================================
# Loads saved pipeline (models + encoders) from training phase
# and predicts SECTION + CLUSTER for new device entries.
#
# Guard rules (evaluated BEFORE model inference):
#   • Unseen / missing CUSTOMER  → BLOCKED  (confidence: N/A)
#   • Unseen / missing PROJECT   → ⚠ WARNING only, still runs
#                                   project_enc set to 0 (neutral)
#                                   numeric_block_rank defaults to 1
#   • Confidence < threshold     → UNKNOWN  (shown with actual %)
## ============================================================

import pandas as pd
import numpy as np
import re
import pickle
import os
import warnings
from sklearn.preprocessing import LabelEncoder
warnings.filterwarnings('ignore')


# ============================================================
# 📂 PATHS
# ============================================================
MODEL_DIR = r"C:\Users\sitisyaziyah\source\repos\DeviceCluster\1.training_model\model_config"

UNKNOWN_THRESHOLD = 0.50   # confidence below this → UNKNOWN

# Sentinel used internally when project is missing/unseen
_NO_PROJECT = "__NO_PROJECT__"


# ============================================================
# 🔐 SafeLabelEncoder
#    Must be defined before pickle.load() is called.
# ============================================================

class SafeLabelEncoder:
    """
    LabelEncoder that handles unseen labels gracefully.
    Adds '__UNKNOWN__' as a sentinel class during fit.
    Unseen values during transform map to '__UNKNOWN__'
    instead of raising a ValueError.
    """

    UNKNOWN_LABEL = "__UNKNOWN__"   # add special class = unknown. handle unseen value during inference

    def __init__(self):
        self._le = LabelEncoder()
        self.classes_ = None

    def fit(self, y):
        labels = list(pd.Series(y).astype(str).unique())
        if self.UNKNOWN_LABEL not in labels:
            labels = [self.UNKNOWN_LABEL] + labels
        self._le.fit(labels)
        self.classes_ = self._le.classes_
        return self

    def transform(self, y):
        y_str = pd.Series(y).astype(str)
        known = set(self.classes_)
        y_safe = y_str.where(y_str.isin(known), other=self.UNKNOWN_LABEL)
        return self._le.transform(y_safe)

    def fit_transform(self, y):
        return self.fit(y).transform(y)

    def inverse_transform(self, y):
        return self._le.inverse_transform(y)

    def is_known(self, values):
        known = set(self.classes_) - {self.UNKNOWN_LABEL}
        return pd.Series(values).astype(str).isin(known)

    def real_classes(self):
        """Return classes_ excluding the __UNKNOWN__ sentinel."""
        return [c for c in self.classes_ if c != self.UNKNOWN_LABEL]


# ============================================================
# 🔧 FEATURE ENGINEERING FUNCTIONS (mirrors training exactly)
# ============================================================

def extract_numeric_block(device_id):
    match = re.search(r'\d+', str(device_id))
    return int(match.group()) if match else -1

def extract_prefix(device_id):
    match = re.match(r'^([A-Za-z]+)', str(device_id))
    return match.group(1).upper() if match else ''

def extract_suffix_letters(device_id):
    match = re.search(r'\d+([A-Za-z]*)$', str(device_id))
    return match.group(1).upper() if match else ''

def extract_suffix_full(device_id):
    match = re.search(r'\d+(.*)$', str(device_id))
    return match.group(1) if match else ''


# ============================================================
# 📦 LOAD PIPELINE
# ============================================================

def load_pipeline(model_dir: str) -> dict:
    """
    Load saved models and encoders from the training phase.

    SafeLabelEncoder must be defined above before this is
    called — pickle needs the class in scope to deserialise.
    """
    print("=" * 80)
    print("📦 LOADING SAVED PIPELINE")
    print("=" * 80)

    model_section   = pickle.load(open(os.path.join(model_dir, "model_section.pkl"),  "rb"))
    model_cluster   = pickle.load(open(os.path.join(model_dir, "model_cluster.pkl"),  "rb"))
    pipeline_config = pickle.load(open(os.path.join(model_dir, "pipeline_config.pkl"), "rb"))

    df_num_path = os.path.join(model_dir, "df_oiltek_num.pkl")
    df_lookup   = pickle.load(open(df_num_path, "rb")) if os.path.exists(df_num_path) else None

    known_customers = pipeline_config['known_customers']
    known_projects  = set(pipeline_config['le_project'].real_classes())

    print("   ✅ model_section.pkl     loaded")
    print("   ✅ model_cluster.pkl     loaded")
    print("   ✅ pipeline_config.pkl   loaded")
    print(f"   {'✅' if df_lookup is not None else '⚠ '} df_oiltek_num.pkl       "
          f"{'loaded' if df_lookup is not None else 'not found — rank will default to 1'}")
    print(f"\n   Known customers : {sorted(known_customers)}")
    print(f"   Known projects  : {sorted(known_projects)}")
    print(f"   Section classes : {pipeline_config['le_section'].real_classes()}")
    print(f"   Cluster classes : {pipeline_config['le_cluster'].real_classes()}")

    return {
        "model_section"   : model_section,
        "model_cluster"   : model_cluster,
        "config"          : pipeline_config,
        "df_lookup"       : df_lookup,
        "known_customers" : known_customers,
        "known_projects"  : known_projects,
    }


# ============================================================
# 🛡️  GUARD + WARNING — VALIDATE ENTITIES BEFORE INFERENCE
# ============================================================

def check_entities(df: pd.DataFrame, pipeline: dict) -> tuple[pd.Series, pd.Series]:
    """
    Returns two Series (indexed like df):

    rejection_reasons : str
        Non-empty → row is BLOCKED, no inference.
        Currently only triggered by unseen/missing CUSTOMER.

    warnings_col : str
        Non-empty → row runs inference WITH a caveat note.
        Triggered by missing or unseen PROJECT.
        Multiple warnings separated by "; ".
    """
    known_customers = pipeline["known_customers"]
    known_projects  = pipeline["known_projects"]

    rejections = []
    warnings   = []

    for _, row in df.iterrows():
        row_reject  = []
        row_warning = []

        # ── CUSTOMER: hard block if unseen/missing ──────────
        cust = str(row.get('CUSTOMER', '')).strip()
        if not cust or cust.upper() in ('', 'NAN', 'NONE'):
            row_reject.append("missing CUSTOMER")
        elif cust not in known_customers:
            row_reject.append(f"unseen CUSTOMER '{cust}'")

        # ── PROJECT: soft warning if unseen/missing ──────────
        proj = str(row.get('PROJECT', '')).strip()
        if not proj or proj.upper() in ('', 'NAN', 'NONE', _NO_PROJECT):
            row_warning.append("PROJECT not provided — prediction uses customer context only")
        elif proj not in known_projects:
            row_warning.append(f"unseen PROJECT '{proj}' — prediction uses customer context only")

        rejections.append("; ".join(row_reject))
        warnings.append("; ".join(row_warning))

    return pd.Series(rejections, index=df.index), pd.Series(warnings, index=df.index)


# ============================================================
# 🔧 BUILD FEATURES (eligible rows only)
# ============================================================

def build_features(df: pd.DataFrame, config: dict,
                   df_lookup, project_warnings: pd.Series) -> pd.DataFrame:
    """
    Constructs the full feature matrix for eligible rows.
    Mirrors the feature engineering in the training script exactly.

    project_warnings : Series aligned to df
        When non-empty for a row, that row's project_enc is set to 0
        and numeric_block_rank defaults to 1 (no lookup possible).
    """
    le_customer    = config['le_customer']
    le_project     = config['le_project']
    le_prefix      = config['le_prefix']
    le_suffix_lt   = config['le_suffix_lt']
    le_suffix_last = config['le_suffix_last']

    df = df.copy().reset_index(drop=True)
    project_warnings = project_warnings.reset_index(drop=True)

    # ── Basic device ID features ─────────────────────────────
    df['numeric_block']         = df['DEVICE_ID'].apply(extract_numeric_block)
    df['device_prefix']         = df['DEVICE_ID'].apply(extract_prefix)
    df['device_suffix_letter']  = df['DEVICE_ID'].apply(extract_suffix_letters)
    df['suffix_full']           = df['DEVICE_ID'].apply(extract_suffix_full)
    df['prefix_length']         = df['device_prefix'].str.len()
    df['device_id_length']      = df['DEVICE_ID'].astype(str).str.len()
    df['has_suffix_letter']     = (df['device_suffix_letter'] != '').astype(int)
    df['has_numeric']           = (df['numeric_block'] != -1).astype(int)

    # ── Advanced suffix features ─────────────────────────────
    df['suffix_length']             = df['suffix_full'].astype(str).str.len()
    df['suffix_has_digit']          = df['suffix_full'].astype(str).str.contains(r'\d', regex=True).astype(int)
    df['suffix_has_letter']         = df['suffix_full'].astype(str).str.contains(r'[A-Za-z]', regex=True).astype(int)
    df['suffix_has_decimal']        = df['suffix_full'].astype(str).str.contains(r'\.', regex=True).astype(int)
    df['suffix_digit_count']        = df['suffix_full'].astype(str).str.count(r'\d')
    df['suffix_letter_count']       = df['suffix_full'].astype(str).str.count(r'[A-Za-z]')
    df['suffix_starts_with_digit']  = df['suffix_full'].astype(str).str[0].apply(lambda c: 1 if isinstance(c, str) and c.isdigit() else 0)
    df['suffix_last_char']          = df['suffix_full'].astype(str).str[-1].fillna('')
    df['suffix_last_char_is_letter']= df['suffix_last_char'].apply(lambda c: 1 if isinstance(c, str) and c.isalpha() else 0)
    df['suffix_last_char_is_digit'] = df['suffix_last_char'].apply(lambda c: 1 if isinstance(c, str) and c.isdigit() else 0)

    # ── Equipment ID features ─────────────────────────────────
    df['equip_id_length']       = df['DEVICE_ID'].astype(str).str.len()
    df['equip_id_digit_count']  = df['DEVICE_ID'].astype(str).str.count(r'\d')
    df['equip_id_letter_count'] = df['DEVICE_ID'].astype(str).str.count(r'[A-Za-z]')

    # ── Numeric block rank ────────────────────────────────────
    # If project is missing/unseen, skip the lookup and use 1.
    # The lookup is still used when project IS known, to replicate
    # the rank that device had during training.
    def get_block_rank(row, has_project_warning: bool):
        if has_project_warning or df_lookup is None:
            return 1
        subset = df_lookup[
            (df_lookup['CUSTOMER'] == row['CUSTOMER']) &
            (df_lookup['PROJECT']  == row['PROJECT'])
        ]
        if subset.empty:
            return 1
        unique_blocks = sorted(subset['numeric_block'].unique())
        block_to_rank = {b: i + 1 for i, b in enumerate(unique_blocks)}
        return block_to_rank.get(row['numeric_block'], 1)

    df['numeric_block_rank'] = [
        get_block_rank(df.iloc[i], bool(project_warnings.iloc[i]))
        for i in range(len(df))
    ]

    # ── Label encoding ────────────────────────────────────────
    # CUSTOMER: guaranteed known at this point (guard passed).
    # PROJECT:  use encoder when known; set to 0 when missing/unseen.
    # prefix/suffix: SafeLabelEncoder maps unseen → __UNKNOWN__ idx.

    df['customer_enc'] = le_customer.transform(df['CUSTOMER'])

    project_enc = []
    for i, row in df.iterrows():
        if project_warnings.iloc[i]:
            # No valid project → neutral encoding (0)
            project_enc.append(0)
        else:
            project_enc.append(int(le_project.transform([row['PROJECT']])[0]))
    df['project_enc'] = project_enc

    df['prefix_enc']           = le_prefix.transform(df['device_prefix'])
    df['suffix_letter_enc']    = le_suffix_lt.transform(df['device_suffix_letter'])
    df['suffix_last_char_enc'] = le_suffix_last.transform(df['suffix_last_char'])

    return df


# ============================================================
# 🚀 PREDICT
# ============================================================

def predict(records: list[dict], pipeline: dict,
            threshold: float = UNKNOWN_THRESHOLD) -> pd.DataFrame:
    """
    Guard → feature engineering → chained model inference.

    Flow
    ----
    1. Normalise column names to uppercase.
       Missing 'project' key is filled with _NO_PROJECT sentinel.
    2. check_entities() separates hard blocks (bad customer) from
       soft warnings (missing/unseen project).
    3. Blocked rows → UNKNOWN / N/A immediately, no inference.
    4. Eligible rows (including those with project warnings) →
       build_features() → two-stage section → cluster chain.
    5. Predictions below `threshold` → replaced with UNKNOWN.

    Returns
    -------
    pd.DataFrame columns:
        DEVICE_ID, CUSTOMER, PROJECT,
        PREDICTED_SECTION, SECTION_CONFIDENCE (%),
        PREDICTED_CLUSTER, CLUSTER_CONFIDENCE (%),
        REJECTION_REASON,  WARNING
    """
    config        = pipeline["config"]
    model_section = pipeline["model_section"]
    model_cluster = pipeline["model_cluster"]
    df_lookup     = pipeline["df_lookup"]
    le_section    = config["le_section"]
    le_cluster    = config["le_cluster"]

    section_features = config["section_features"]
    cluster_features = config["cluster_features"]

    # ── Normalise input ───────────────────────────────────────
    base_df = pd.DataFrame(records)
    base_df.columns = base_df.columns.str.upper()

    # Fill missing PROJECT column or NaN values with sentinel
    if 'PROJECT' not in base_df.columns:
        base_df['PROJECT'] = _NO_PROJECT
    else:
        base_df['PROJECT'] = base_df['PROJECT'].fillna(_NO_PROJECT).astype(str).str.strip()
        base_df['PROJECT'] = base_df['PROJECT'].replace({'': _NO_PROJECT, 'nan': _NO_PROJECT, 'None': _NO_PROJECT})

    # ── Guard + warnings ──────────────────────────────────────
    rejection_reasons, warnings_col = check_entities(base_df, pipeline)
    eligible_mask = rejection_reasons == ""

    # ── Pre-fill outputs ──────────────────────────────────────
    pred_section = ["UNKNOWN"] * len(base_df)
    sec_conf     = [None]      * len(base_df)    # None = blocked (not evaluated)
    pred_cluster = ["UNKNOWN"] * len(base_df)
    clu_conf     = [None]      * len(base_df)

    # ── Inference on eligible rows ────────────────────────────
    if eligible_mask.any():
        elig_idx      = base_df.index[eligible_mask].tolist()
        df_elig       = base_df.loc[elig_idx].copy()
        proj_warn_elig = warnings_col.loc[elig_idx].copy()

        df_feat = build_features(df_elig, config, df_lookup, proj_warn_elig)

        X_sec = (
            df_feat[section_features]
            .apply(pd.to_numeric, errors='coerce')
            .fillna(0)
        )

        # Stage 1 — Section
        sec_proba_raw = model_section.predict_proba(X_sec)
        sec_pred_enc  = np.argmax(sec_proba_raw, axis=1)
        sec_conf_raw  = sec_proba_raw.max(axis=1)

        sec_decoded = le_section.inverse_transform(sec_pred_enc)
        sec_decoded = np.where(
            sec_decoded == SafeLabelEncoder.UNKNOWN_LABEL, "UNKNOWN", sec_decoded
        )

        # Chain — Cluster
        X_clu = (
            df_feat[[f for f in cluster_features if f != "predicted_section"]]
            .apply(pd.to_numeric, errors='coerce')
            .fillna(0)
            .copy()
        )
        X_clu["predicted_section"] = sec_pred_enc
        X_clu = X_clu[cluster_features]

        # Stage 2 — Cluster
        clu_proba_raw = model_cluster.predict_proba(X_clu)
        clu_pred_enc  = np.argmax(clu_proba_raw, axis=1)
        clu_conf_raw  = clu_proba_raw.max(axis=1)

        clu_decoded = le_cluster.inverse_transform(clu_pred_enc)
        clu_decoded = np.where(
            clu_decoded == SafeLabelEncoder.UNKNOWN_LABEL, "UNKNOWN", clu_decoded
        )

        # Apply confidence threshold
        sec_final = np.where(sec_conf_raw >= threshold, sec_decoded, "UNKNOWN")
        clu_final = np.where(clu_conf_raw >= threshold, clu_decoded, "UNKNOWN")

        # Write back to original positions
        for i, orig_idx in enumerate(elig_idx):
            pred_section[orig_idx] = sec_final[i]
            sec_conf[orig_idx]     = round(float(sec_conf_raw[i]) * 100, 2)
            pred_cluster[orig_idx] = clu_final[i]
            clu_conf[orig_idx]     = round(float(clu_conf_raw[i]) * 100, 2)

    # ── Assemble result ───────────────────────────────────────
    # Show _NO_PROJECT sentinel as empty string in output
    display_project = base_df['PROJECT'].replace(_NO_PROJECT, "(not provided)")

    result = base_df[["DEVICE_ID", "CUSTOMER"]].copy()
    result["PROJECT"]            = display_project
    result["PREDICTED_SECTION"]  = pred_section
    result["SECTION_CONFIDENCE"] = sec_conf
    result["PREDICTED_CLUSTER"]  = pred_cluster
    result["CLUSTER_CONFIDENCE"] = clu_conf
    result["REJECTION_REASON"]   = rejection_reasons.values
    result["WARNING"]            = warnings_col.values

    return result


# ============================================================
# 🖨️  PRETTY PRINT RESULTS
# ============================================================

def print_results(result: pd.DataFrame):
    print("\n" + "=" * 80)
    print("📊 PREDICTION RESULTS")
    print("=" * 80)

    for _, row in result.iterrows():
        rejected = bool(row["REJECTION_REASON"])
        noted = bool(row["PROJECT_NOTE"])

        proj_display = row["PROJECT"] if row["PROJECT"] else "(not provided)"
        print(f"\n   Device   : {row['DEVICE_ID']}")
        print(f"   Customer : {row['CUSTOMER']}   |   Project : {proj_display}")

        if rejected:
            # Hard block — no inference ran
            print(f"   🚫 BLOCKED  — {row['REJECTION_REASON']}")
            print(f"   ⛔ Section : {'UNKNOWN':20s}  (confidence: N/A — not evaluated)")
            print(f"   ⛔ Cluster : {'UNKNOWN':20s}  (confidence: N/A — not evaluated)")

        else:
            if noted:
                # Soft warning — inference ran with reduced context
                print(f"   ℹ️  {row['PROJECT_NOTE']}")
                
            sec_unk      = row["PREDICTED_SECTION"] == "UNKNOWN"
            clu_unk      = row["PREDICTED_CLUSTER"] == "UNKNOWN"
            sec_icon     = "⚠" if sec_unk else "✅"
            clu_icon     = "⚠" if clu_unk else "✅"
            sec_conf_str = f"{row['SECTION_CONFIDENCE']:.1f}%" if row["SECTION_CONFIDENCE"] is not None else "N/A"
            clu_conf_str = f"{row['CLUSTER_CONFIDENCE']:.1f}%" if row["CLUSTER_CONFIDENCE"] is not None else "N/A"

            print(f"   {sec_icon} Section : {row['PREDICTED_SECTION']:20s}  (confidence: {sec_conf_str})")
            print(f"   {clu_icon} Cluster : {row['PREDICTED_CLUSTER']:20s}  (confidence: {clu_conf_str})")

        print(f"   {'-' * 60}")

    # ── Summary ───────────────────────────────────────────────
    total     = len(result)
    blocked   = (result["REJECTION_REASON"] != "").sum()
    warned    = ((result["WARNING"] != "") & (result["REJECTION_REASON"] == "")).sum()
    evaluated = total - blocked
    unk_sec   = ((result["PREDICTED_SECTION"] == "UNKNOWN") & (result["REJECTION_REASON"] == "")).sum()
    unk_clu   = ((result["PREDICTED_CLUSTER"] == "UNKNOWN") & (result["REJECTION_REASON"] == "")).sum()

    print(f"\n{'=' * 80}")
    print(f"   Total records              : {total}")
    print(f"   Evaluated by model         : {evaluated}   "
          f"(of which ⚠ project-less: {warned})   |   Blocked: {blocked}")
    if unk_sec or unk_clu:
        print(f"   Low-confidence UNKNOWN  →  Section : {unk_sec}   |   Cluster : {unk_clu}")
    print(f"   Confidence threshold used  : {UNKNOWN_THRESHOLD * 100:.0f}%")
    print("=" * 80)


# ============================================================
# ▶️  MAIN — EXAMPLE APPLICATION
# ============================================================

if __name__ == "__main__":

    test_records = [
    {"device_id": "LL510", "customer": "OILTEK", "project": "A1706"},
    {"device_id": "LL510", "customer": "LIPICO", "project": "A1706"},
    {"device_id": "LL510", "customer": "OILTEK", "project": "A1710"},
    ]

    # 1. Load pipeline
    pipeline = load_pipeline(MODEL_DIR)

    # 2. Predict
    result_df = predict(test_records, pipeline, threshold=UNKNOWN_THRESHOLD)

    # 3. Print
    print_results(result_df)

    # 4. Export
    out_path = os.path.join(MODEL_DIR, "application_results.xlsx")
    result_df.to_excel(out_path, index=False)
    print(f"\n   💾 Results saved to: {out_path}")


# ============================================================
# 🔁  SINGLE-DEVICE CONVENIENCE FUNCTION
# ============================================================

def predict_single(device_id: str, customer: str, project: str = None,
                   pipeline: dict = None, threshold: float = UNKNOWN_THRESHOLD) -> dict:
    """
    Shorthand for predicting one device at a time.
    `project` is optional — omit or pass None to predict
    using customer context only.

    Example
    -------
    >>> pipeline = load_pipeline(MODEL_DIR)
    >>> # With project:
    >>> result = predict_single("LL510", "OILTEK", "A1706", pipeline)
    >>> # Without project:
    >>> result = predict_single("LL510", "OILTEK", pipeline=pipeline)
    >>> print(result)
    """
    record = {"device_id": device_id, "customer": customer}
    if project is not None:
        record["project"] = project

    result = predict([record], pipeline, threshold=threshold)
    row = result.iloc[0]
    return {
        "device_id"          : row["DEVICE_ID"],
        "customer"           : row["CUSTOMER"],
        "project"            : row["PROJECT"],
        "predicted_section"  : row["PREDICTED_SECTION"],
        "section_confidence" : row["SECTION_CONFIDENCE"],   # None if blocked
        "predicted_cluster"  : row["PREDICTED_CLUSTER"],
        "cluster_confidence" : row["CLUSTER_CONFIDENCE"],   # None if blocked
        "rejection_reason"   : row["REJECTION_REASON"],
        "warning"            : row["WARNING"],
    }

📦 LOADING SAVED PIPELINE
   ✅ model_section.pkl     loaded
   ✅ model_cluster.pkl     loaded
   ✅ pipeline_config.pkl   loaded
   ✅ df_oiltek_num.pkl       loaded

   Known customers : [np.str_('OILTEK'), np.str_('UGS')]
   Known projects  : [np.str_('A1706'), np.str_('A1993'), np.str_('A1994'), np.str_('A1995'), np.str_('A2027'), np.str_('A8881'), np.str_('A8882'), np.str_('A8883'), np.str_('A8884'), np.str_('A8885'), np.str_('A9991'), np.str_('A9992')]
   Section classes : [np.str_('SECTION 1'), np.str_('SECTION 2'), np.str_('SECTION 3'), np.str_('SECTION 4'), np.str_('SECTION 5'), np.str_('SECTION 6')]
   Cluster classes : [np.str_('CLUSTER 1'), np.str_('CLUSTER 2'), np.str_('CLUSTER 3'), np.str_('CLUSTER 4'), np.str_('CLUSTER 5'), np.str_('CLUSTER 6'), np.str_('CLUSTER 7'), np.str_('CLUSTER 8'), np.str_('CLUSTER 9')]

📊 PREDICTION RESULTS


KeyError: 'PROJECT_NOTE'

In [ ]:
# check pickle file for this 

pipeline = load_pipeline(MODEL_DIR)

le_customer = pipeline["config"]["le_customer"]
print(le_customer.classes_)
print(le_customer.real_classes())

📦 LOADING SAVED PIPELINE
   ✅ model_section.pkl     loaded
   ✅ model_cluster.pkl     loaded
   ✅ pipeline_config.pkl   loaded
   ✅ df_oiltek_num.pkl       loaded

   Known customers : [np.str_('OILTEK'), np.str_('UGS')]
   Known projects  : [np.str_('A1706'), np.str_('A1993'), np.str_('A1994'), np.str_('A1995'), np.str_('A2027'), np.str_('A8881'), np.str_('A8882'), np.str_('A8883'), np.str_('A8884'), np.str_('A8885'), np.str_('A9991'), np.str_('A9992')]
   Section classes : [np.str_('SECTION 1'), np.str_('SECTION 2'), np.str_('SECTION 3'), np.str_('SECTION 4'), np.str_('SECTION 5'), np.str_('SECTION 6')]
   Cluster classes : [np.str_('CLUSTER 1'), np.str_('CLUSTER 2'), np.str_('CLUSTER 3'), np.str_('CLUSTER 4'), np.str_('CLUSTER 5'), np.str_('CLUSTER 6'), np.str_('CLUSTER 7'), np.str_('CLUSTER 8'), np.str_('CLUSTER 9')]
['OILTEK' 'UGS' '__UNKNOWN__']
[np.str_('OILTEK'), np.str_('UGS')]


# Edited script [24/4 16:00] - current

In [4]:
# ============================================================
# 🤖 DEVICE CLUSTER — MODEL APPLICATION SCRIPT
# ============================================================
# Guard rules:
#   • Unseen / missing CUSTOMER  → BLOCKED  (confidence: N/A)
#   • Unseen / missing PROJECT   → silently falls back to
#                                   customer-level aggregation
#                                   (mean project_enc + mean rank)
#   • Confidence < threshold     → UNKNOWN  (shown with actual %)
# ============================================================

import pandas as pd
import numpy as np
import re
import pickle
import os
import warnings
from sklearn.preprocessing import LabelEncoder
warnings.filterwarnings('ignore')


# ============================================================
# 📂 PATHS
# ============================================================
MODEL_DIR = r"C:\Users\sitisyaziyah\source\repos\DeviceCluster\1.training_model\model_config"

UNKNOWN_THRESHOLD = 0.50

_NO_PROJECT = "__NO_PROJECT__"


# ============================================================
# 🔐 SafeLabelEncoder
# ============================================================

class SafeLabelEncoder:
    UNKNOWN_LABEL = "__UNKNOWN__"

    def __init__(self):
        self._le = LabelEncoder()
        self.classes_ = None

    def fit(self, y):
        labels = list(pd.Series(y).astype(str).unique())
        if self.UNKNOWN_LABEL not in labels:
            labels = [self.UNKNOWN_LABEL] + labels
        self._le.fit(labels)
        self.classes_ = self._le.classes_
        return self

    def transform(self, y):
        y_str = pd.Series(y).astype(str)
        known = set(self.classes_)
        y_safe = y_str.where(y_str.isin(known), other=self.UNKNOWN_LABEL)
        return self._le.transform(y_safe)

    def fit_transform(self, y):
        return self.fit(y).transform(y)

    def inverse_transform(self, y):
        return self._le.inverse_transform(y)

    def is_known(self, values):
        known = set(self.classes_) - {self.UNKNOWN_LABEL}
        return pd.Series(values).astype(str).isin(known)

    def real_classes(self):
        return [c for c in self.classes_ if c != self.UNKNOWN_LABEL]


# ============================================================
# 🔧 FEATURE ENGINEERING FUNCTIONS
# ============================================================

def extract_numeric_block(device_id):
    match = re.search(r'\d+', str(device_id))
    return int(match.group()) if match else -1

def extract_prefix(device_id):
    match = re.match(r'^([A-Za-z]+)', str(device_id))
    return match.group(1).upper() if match else ''

def extract_suffix_letters(device_id):
    match = re.search(r'\d+([A-Za-z]*)$', str(device_id))
    return match.group(1).upper() if match else ''

def extract_suffix_full(device_id):
    match = re.search(r'\d+(.*)$', str(device_id))
    return match.group(1) if match else ''


# ============================================================
# 📦 LOAD PIPELINE
# ============================================================

def load_pipeline(model_dir: str) -> dict:
    print("=" * 80)
    print("📦 LOADING SAVED PIPELINE")
    print("=" * 80)

    model_section   = pickle.load(open(os.path.join(model_dir, "model_section.pkl"),  "rb"))
    model_cluster   = pickle.load(open(os.path.join(model_dir, "model_cluster.pkl"),  "rb"))
    pipeline_config = pickle.load(open(os.path.join(model_dir, "pipeline_config.pkl"), "rb"))

    df_num_path = os.path.join(model_dir, "df_oiltek_num.pkl")
    df_lookup   = pickle.load(open(df_num_path, "rb")) if os.path.exists(df_num_path) else None

    known_customers = pipeline_config['known_customers']
    known_projects  = set(pipeline_config['le_project'].real_classes())

    print("   ✅ model_section.pkl     loaded")
    print("   ✅ model_cluster.pkl     loaded")
    print("   ✅ pipeline_config.pkl   loaded")
    print(f"   {'✅' if df_lookup is not None else '⚠ '} df_oiltek_num.pkl       "
          f"{'loaded' if df_lookup is not None else 'not found — rank will default to customer mean'}")
    print(f"\n   Known customers : {sorted(known_customers)}")
    print(f"   Known projects  : {sorted(known_projects)}")
    print(f"   Section classes : {pipeline_config['le_section'].real_classes()}")
    print(f"   Cluster classes : {pipeline_config['le_cluster'].real_classes()}")

    # ── Pre-compute per-customer aggregates ──────────────────
    # For rows where project is unknown, we use the mean project_enc
    # and mean numeric_block_rank across all known projects of that customer.
    # This keeps the feature distribution realistic rather than forcing zeros.
    customer_project_agg = {}
    if df_lookup is not None:
        le_project = pipeline_config['le_project']
        for customer, grp in df_lookup.groupby('CUSTOMER'):
            known_proj_in_grp = [
                p for p in grp['PROJECT'].unique()
                if p in known_projects
            ]
            if not known_proj_in_grp:
                customer_project_agg[customer] = {"mean_project_enc": 0, "mean_rank": 1}
                continue

            # Mean project encoding across this customer's projects
            proj_encs = [
                int(le_project.transform([p])[0])
                for p in known_proj_in_grp
            ]

            # Mean numeric_block_rank across this customer's projects
            ranks = []
            for proj in known_proj_in_grp:
                subset = grp[grp['PROJECT'] == proj]
                unique_blocks = sorted(subset['numeric_block'].unique())
                block_to_rank = {b: i + 1 for i, b in enumerate(unique_blocks)}
                ranks.extend(block_to_rank.values())

            customer_project_agg[customer] = {
                "mean_project_enc": round(np.mean(proj_encs)),   # round to nearest valid int
                "mean_rank"       : round(np.mean(ranks), 2),
            }

    return {
        "model_section"        : model_section,
        "model_cluster"        : model_cluster,
        "config"               : pipeline_config,
        "df_lookup"            : df_lookup,
        "known_customers"      : known_customers,
        "known_projects"       : known_projects,
        "customer_project_agg" : customer_project_agg,   # ← new
    }


# ============================================================
# 🛡️  GUARD — VALIDATE CUSTOMER; NOTE NEW PROJECTS
# ============================================================

def check_entities(df: pd.DataFrame, pipeline: dict) -> tuple[pd.Series, pd.Series]:
    """
    Returns:
        rejection_reasons : hard block — triggered only by unseen/missing CUSTOMER
        project_notes     : informational only — triggered when project is new/missing
                            Inference still runs at full quality using customer average.
    """
    known_customers = pipeline["known_customers"]
    known_projects  = pipeline["known_projects"]

    rejections    = []
    project_notes = []

    for _, row in df.iterrows():
        row_reject = []
        row_note   = []

        # ── CUSTOMER: hard block if unseen/missing ───────────
        cust = str(row.get('CUSTOMER', '')).strip()
        if not cust or cust.upper() in ('', 'NAN', 'NONE'):
            row_reject.append("missing CUSTOMER")
        elif cust not in known_customers:
            row_reject.append(f"unseen CUSTOMER '{cust}' — please assign manually")

        # ── PROJECT: informational note only ─────────────────
        proj = str(row.get('PROJECT', '')).strip()
        if not proj or proj.upper() in ('', 'NAN', 'NONE', _NO_PROJECT):
            row_note.append("no project provided")
        elif proj not in known_projects:
            row_note.append(f"new project detected: '{proj}'")
        # Known project → no note needed

        rejections.append("; ".join(row_reject))
        project_notes.append("; ".join(row_note))

    return (
        pd.Series(rejections,    index=df.index),
        pd.Series(project_notes, index=df.index),
    )


# ============================================================
# 🔧 BUILD FEATURES
# ============================================================

def build_features(df: pd.DataFrame, config: dict, df_lookup,
                   known_projects: set, customer_project_agg: dict) -> pd.DataFrame:
    """
    Mirrors training feature engineering exactly.

    Unknown / missing project → uses customer-level mean
    project_enc and mean numeric_block_rank instead of zeros,
    keeping prediction quality on par with known-project rows.
    """
    le_customer    = config['le_customer']
    le_project     = config['le_project']
    le_prefix      = config['le_prefix']
    le_suffix_lt   = config['le_suffix_lt']
    le_suffix_last = config['le_suffix_last']

    df = df.copy().reset_index(drop=True)

    # ── Basic device ID features ─────────────────────────────
    df['numeric_block']        = df['DEVICE_ID'].apply(extract_numeric_block)
    df['device_prefix']        = df['DEVICE_ID'].apply(extract_prefix)
    df['device_suffix_letter'] = df['DEVICE_ID'].apply(extract_suffix_letters)
    df['suffix_full']          = df['DEVICE_ID'].apply(extract_suffix_full)
    df['prefix_length']        = df['device_prefix'].str.len()
    df['device_id_length']     = df['DEVICE_ID'].astype(str).str.len()
    df['has_suffix_letter']    = (df['device_suffix_letter'] != '').astype(int)
    df['has_numeric']          = (df['numeric_block'] != -1).astype(int)

    # ── Advanced suffix features ─────────────────────────────
    df['suffix_length']             = df['suffix_full'].astype(str).str.len()
    df['suffix_has_digit']          = df['suffix_full'].astype(str).str.contains(r'\d', regex=True).astype(int)
    df['suffix_has_letter']         = df['suffix_full'].astype(str).str.contains(r'[A-Za-z]', regex=True).astype(int)
    df['suffix_has_decimal']        = df['suffix_full'].astype(str).str.contains(r'\.', regex=True).astype(int)
    df['suffix_digit_count']        = df['suffix_full'].astype(str).str.count(r'\d')
    df['suffix_letter_count']       = df['suffix_full'].astype(str).str.count(r'[A-Za-z]')
    df['suffix_starts_with_digit']  = df['suffix_full'].astype(str).str[0].apply(lambda c: 1 if isinstance(c, str) and c.isdigit() else 0)
    df['suffix_last_char']          = df['suffix_full'].astype(str).str[-1].fillna('')
    df['suffix_last_char_is_letter']= df['suffix_last_char'].apply(lambda c: 1 if isinstance(c, str) and c.isalpha() else 0)
    df['suffix_last_char_is_digit'] = df['suffix_last_char'].apply(lambda c: 1 if isinstance(c, str) and c.isdigit() else 0)

    # ── Equipment ID features ─────────────────────────────────
    df['equip_id_length']      = df['DEVICE_ID'].astype(str).str.len()
    df['equip_id_digit_count'] = df['DEVICE_ID'].astype(str).str.count(r'\d')
    df['equip_id_letter_count']= df['DEVICE_ID'].astype(str).str.count(r'[A-Za-z]')

    # ── Project validity flag per row ─────────────────────────
    def project_is_known(row):
        proj = str(row.get('PROJECT', '')).strip()
        return proj not in ('', 'nan', 'None', _NO_PROJECT) and proj in known_projects

    df['_project_known'] = df.apply(project_is_known, axis=1)

    # ── Numeric block rank ────────────────────────────────────
    def get_block_rank(row):
        cust = row['CUSTOMER']
        if not row['_project_known'] or df_lookup is None:
            # Unknown project → use customer-level mean rank
            agg = customer_project_agg.get(cust, {})
            return agg.get("mean_rank", 1)
        subset = df_lookup[
            (df_lookup['CUSTOMER'] == cust) &
            (df_lookup['PROJECT']  == row['PROJECT'])
        ]
        if subset.empty:
            return customer_project_agg.get(cust, {}).get("mean_rank", 1)
        unique_blocks = sorted(subset['numeric_block'].unique())
        block_to_rank = {b: i + 1 for i, b in enumerate(unique_blocks)}
        return block_to_rank.get(row['numeric_block'], 1)

    df['numeric_block_rank'] = df.apply(get_block_rank, axis=1)

    # ── Label encoding ────────────────────────────────────────
    df['customer_enc'] = le_customer.transform(df['CUSTOMER'])

    def get_project_enc(row):
        if row['_project_known']:
            return int(le_project.transform([row['PROJECT']])[0])
        # Unknown project → customer-level mean project encoding
        agg = customer_project_agg.get(row['CUSTOMER'], {})
        return agg.get("mean_project_enc", 0)

    df['project_enc']          = df.apply(get_project_enc, axis=1)
    df['prefix_enc']           = le_prefix.transform(df['device_prefix'])
    df['suffix_letter_enc']    = le_suffix_lt.transform(df['device_suffix_letter'])
    df['suffix_last_char_enc'] = le_suffix_last.transform(df['suffix_last_char'])

    df.drop(columns=['_project_known'], inplace=True)
    return df


# ============================================================
# 🚀 PREDICT
# ============================================================

def predict(records: list[dict], pipeline: dict,
            threshold: float = UNKNOWN_THRESHOLD) -> pd.DataFrame:

    config        = pipeline["config"]
    model_section = pipeline["model_section"]
    model_cluster = pipeline["model_cluster"]
    df_lookup     = pipeline["df_lookup"]
    le_section    = config["le_section"]
    le_cluster    = config["le_cluster"]
    known_projects       = pipeline["known_projects"]
    customer_project_agg = pipeline["customer_project_agg"]

    section_features = config["section_features"]
    cluster_features = config["cluster_features"]

    # ── Normalise input ───────────────────────────────────────
    base_df = pd.DataFrame(records)
    base_df.columns = base_df.columns.str.upper()

    if 'PROJECT' not in base_df.columns:
        base_df['PROJECT'] = _NO_PROJECT
    else:
        base_df['PROJECT'] = base_df['PROJECT'].fillna(_NO_PROJECT).astype(str).str.strip()
        base_df['PROJECT'] = base_df['PROJECT'].replace({'': _NO_PROJECT, 'nan': _NO_PROJECT, 'None': _NO_PROJECT})

    # ── Guard (customer only) ─────────────────────────────────
    rejection_reasons, project_notes_col = check_entities(base_df, pipeline)
    eligible_mask     = rejection_reasons == ""

    # ── Pre-fill outputs ──────────────────────────────────────
    pred_section = ["UNKNOWN"] * len(base_df)
    sec_conf     = [None]      * len(base_df)
    pred_cluster = ["UNKNOWN"] * len(base_df)
    clu_conf     = [None]      * len(base_df)
    project_note = [""] * len(base_df)   # informational only

    # ── Tag rows where project was not provided / unknown ─────
    for i, row in base_df.iterrows():
        proj = row['PROJECT']
        if proj == _NO_PROJECT:
            project_note[i] = "project not provided"
        elif proj not in known_projects:
            project_note[i] = f"unseen project '{proj}'"

    # ── Inference on eligible rows ────────────────────────────
    if eligible_mask.any():
        elig_idx = base_df.index[eligible_mask].tolist()
        df_elig  = base_df.loc[elig_idx].copy()

        df_feat = build_features(
            df_elig, config, df_lookup,
            known_projects, customer_project_agg
        )

        X_sec = (
            df_feat[section_features]
            .apply(pd.to_numeric, errors='coerce')
            .fillna(0)
        )

        # Stage 1 — Section
        sec_proba_raw = model_section.predict_proba(X_sec)
        sec_pred_enc  = np.argmax(sec_proba_raw, axis=1)
        sec_conf_raw  = sec_proba_raw.max(axis=1)

        sec_decoded = le_section.inverse_transform(sec_pred_enc)
        sec_decoded = np.where(sec_decoded == SafeLabelEncoder.UNKNOWN_LABEL, "UNKNOWN", sec_decoded)

        # Chain — Cluster
        X_clu = (
            df_feat[[f for f in cluster_features if f != "predicted_section"]]
            .apply(pd.to_numeric, errors='coerce')
            .fillna(0)
            .copy()
        )
        X_clu["predicted_section"] = sec_pred_enc
        X_clu = X_clu[cluster_features]

        # Stage 2 — Cluster
        clu_proba_raw = model_cluster.predict_proba(X_clu)
        clu_pred_enc  = np.argmax(clu_proba_raw, axis=1)
        clu_conf_raw  = clu_proba_raw.max(axis=1)

        clu_decoded = le_cluster.inverse_transform(clu_pred_enc)
        clu_decoded = np.where(clu_decoded == SafeLabelEncoder.UNKNOWN_LABEL, "UNKNOWN", clu_decoded)

        sec_final = np.where(sec_conf_raw >= threshold, sec_decoded, "UNKNOWN")
        clu_final = np.where(clu_conf_raw >= threshold, clu_decoded, "UNKNOWN")

        for i, orig_idx in enumerate(elig_idx):
            pred_section[orig_idx] = sec_final[i]
            sec_conf[orig_idx]     = round(float(sec_conf_raw[i]) * 100, 2)
            pred_cluster[orig_idx] = clu_final[i]
            clu_conf[orig_idx]     = round(float(clu_conf_raw[i]) * 100, 2)

    # ── Assemble result ───────────────────────────────────────
    display_project = base_df['PROJECT'].replace(_NO_PROJECT, "(not provided)")

    result = base_df[["DEVICE_ID", "CUSTOMER"]].copy()
    result["PROJECT"]            = display_project
    result["PREDICTED_SECTION"]  = pred_section
    result["SECTION_CONFIDENCE"] = sec_conf
    result["PREDICTED_CLUSTER"]  = pred_cluster
    result["CLUSTER_CONFIDENCE"] = clu_conf
    result["REJECTION_REASON"]   = rejection_reasons.values
    result["PROJECT_NOTE"]     = project_notes_col.values
    result["PROJECT_NOTE"]       = project_note   # replaces WARNING

    return result


# ============================================================
# 🖨️  PRETTY PRINT RESULTS
# ============================================================

def print_results(result: pd.DataFrame):
    print("\n" + "=" * 80)
    print("📊 PREDICTION RESULTS")
    print("=" * 80)

    for _, row in result.iterrows():
        rejected = bool(row["REJECTION_REASON"])
        noted    = bool(row["PROJECT_NOTE"])

        proj_display = row["PROJECT"] if row["PROJECT"] else "(not provided)"
        print(f"\n   Device   : {row['DEVICE_ID']}")
        print(f"   Customer : {row['CUSTOMER']}   |   Project : {proj_display}")

        if rejected:
            print(f"   🚫 BLOCKED  — {row['REJECTION_REASON']}")
            print(f"   ⛔ Section : {'UNKNOWN':20s}  (confidence: N/A — not evaluated)")
            print(f"   ⛔ Cluster : {'UNKNOWN':20s}  (confidence: N/A — not evaluated)")
        else:
            if noted:
                print(f"   ❗  NOTE — {row['PROJECT_NOTE']}")

            sec_unk      = row["PREDICTED_SECTION"] == "UNKNOWN"
            clu_unk      = row["PREDICTED_CLUSTER"] == "UNKNOWN"
            sec_icon     = "⚠" if sec_unk else "✅"
            clu_icon     = "⚠" if clu_unk else "✅"
            sec_conf_str = f"{row['SECTION_CONFIDENCE']:.1f}%" if row["SECTION_CONFIDENCE"] is not None else "N/A"
            clu_conf_str = f"{row['CLUSTER_CONFIDENCE']:.1f}%" if row["CLUSTER_CONFIDENCE"] is not None else "N/A"

            print(f"   {sec_icon} Section : {row['PREDICTED_SECTION']:20s}  (confidence: {sec_conf_str})")
            print(f"   {clu_icon} Cluster : {row['PREDICTED_CLUSTER']:20s}  (confidence: {clu_conf_str})")

        print(f"   {'-' * 60}")

    total     = len(result)
    blocked   = (result["REJECTION_REASON"] != "").sum()
    noted     = ((result["PROJECT_NOTE"] != "") & (result["REJECTION_REASON"] == "")).sum()
    evaluated = total - blocked
    unk_sec   = ((result["PREDICTED_SECTION"] == "UNKNOWN") & (result["REJECTION_REASON"] == "")).sum()
    unk_clu   = ((result["PREDICTED_CLUSTER"] == "UNKNOWN") & (result["REJECTION_REASON"] == "")).sum()

    print(f"\n{'=' * 80}")
    print(f"   Total records              : {total}")
    print(f"   Evaluated by model         : {evaluated}   "
          f"(of which ℹ️  customer-avg project: {noted})   |   Blocked: {blocked}")
    if unk_sec or unk_clu:
        print(f"   Low-confidence UNKNOWN  →  Section : {unk_sec}   |   Cluster : {unk_clu}")
    print(f"   Confidence threshold used  : {UNKNOWN_THRESHOLD * 100:.0f}%")
    print("=" * 80)


# ============================================================
# ▶️  MAIN
# ============================================================

if __name__ == "__main__":

    test_records = [
        {"device_id": "LL510", "customer": "OILTEK", "project": "A1706"},   # known project
        
        {"device_id": "LL510", "customer": "OILTEK", "project": "A1710"},   # unknown project → customer avg
        
        {"device_id": "LL510", "customer": "OILTEK"},                       # no project → customer avg
        {"device_id": "LL510", "customer": "LIPICO", "project": "A1706"},   # unknown customer → blocked
    ]

    pipeline  = load_pipeline(MODEL_DIR)
    result_df = predict(test_records, pipeline, threshold=UNKNOWN_THRESHOLD)
    print_results(result_df)

    out_path = os.path.join(MODEL_DIR, "application_results.xlsx")
    result_df.to_excel(out_path, index=False)
    print(f"\n   💾 Results saved to: {out_path}")


# ============================================================
# 🔁  SINGLE-DEVICE CONVENIENCE FUNCTION
# ============================================================

def predict_single(device_id: str, customer: str, project: str = None,
                   pipeline: dict = None, threshold: float = UNKNOWN_THRESHOLD) -> dict:
    record = {"device_id": device_id, "customer": customer}
    if project is not None:
        record["project"] = project

    result = predict([record], pipeline, threshold=threshold)
    row = result.iloc[0]
    return {
        "device_id"          : row["DEVICE_ID"],
        "customer"           : row["CUSTOMER"],
        "project"            : row["PROJECT"],
        "predicted_section"  : row["PREDICTED_SECTION"],
        "section_confidence" : row["SECTION_CONFIDENCE"],
        "predicted_cluster"  : row["PREDICTED_CLUSTER"],
        "cluster_confidence" : row["CLUSTER_CONFIDENCE"],
        "rejection_reason"   : row["REJECTION_REASON"],
        "project_note"       : row["PROJECT_NOTE"],
    }

📦 LOADING SAVED PIPELINE
   ✅ model_section.pkl     loaded
   ✅ model_cluster.pkl     loaded
   ✅ pipeline_config.pkl   loaded
   ✅ df_oiltek_num.pkl       loaded

   Known customers : [np.str_('OILTEK'), np.str_('UGS')]
   Known projects  : [np.str_('A1706'), np.str_('A1993'), np.str_('A1994'), np.str_('A1995'), np.str_('A2027'), np.str_('A8881'), np.str_('A8882'), np.str_('A8883'), np.str_('A8884'), np.str_('A8885'), np.str_('A9991'), np.str_('A9992')]
   Section classes : [np.str_('SECTION 1'), np.str_('SECTION 2'), np.str_('SECTION 3'), np.str_('SECTION 4'), np.str_('SECTION 5'), np.str_('SECTION 6')]
   Cluster classes : [np.str_('CLUSTER 1'), np.str_('CLUSTER 2'), np.str_('CLUSTER 3'), np.str_('CLUSTER 4'), np.str_('CLUSTER 5'), np.str_('CLUSTER 6'), np.str_('CLUSTER 7'), np.str_('CLUSTER 8'), np.str_('CLUSTER 9')]

📊 PREDICTION RESULTS

   Device   : LL510
   Customer : OILTEK   |   Project : A1706
   ✅ Section : SECTION 1             (confidence: 78.0%)
   ✅ Cluster : CLUSTER 1